<a href="https://colab.research.google.com/github/amalfj5-stack/BreastCancer/blob/main/TCGA_BRCA_IDC_Multimodal_Pipeline_CLEAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Section 1 — install packages

In [ ]:
# ============================================================
# 1A) INSTALL SYSTEM DEPENDENCY (OpenSlide native library)
# ============================================================
import subprocess, sys
subprocess.check_call(["apt-get", "install", "-y", "-q", "openslide-tools"], stdout=subprocess.DEVNULL)

# ============================================================
# 1B) INSTALL PYTHON PACKAGES
# ============================================================
import importlib.util

PACKAGE_TO_MODULE = {
    "timm":            "timm",
    "captum":          "captum",
    "shap":            "shap",
    "scikit-image":    "skimage",
    "lime":            "lime",
    "openslide-python":"openslide",
    "requests":        "requests",
}

missing = [pkg for pkg, mod in PACKAGE_TO_MODULE.items()
           if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
else:
    print("All packages already installed.")

In [ ]:
# ============================================================
# 1C) PIN NUMPY / PANDAS VERSIONS
# ============================================================
# Run this cell, then go to Runtime → Restart session, then run all from top.
!pip install -q --force-reinstall --no-cache-dir \
    "numpy==1.26.4" \
    "pandas==2.2.2" \
    "numba==0.60.0" \
    "shap==0.46.0"


## Section 2 — imports, seed, device, and settings
Class names map to IDC (Invasive Ductal Carcinoma) patch classification:
| ID | Class | Definition |
|----|-------|-----------|
| 0 | IDC_Negative | Non-tumour patch |
| 1 | IDC_Positive | IDC tumour patch |


In [ ]:
# ============================================================
# 2) IMPORTS, SEED, DEVICE, AND GLOBAL SETTINGS
# ============================================================
import os, gc, math, time, random, warnings, requests, zipfile, shutil
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score, log_loss, roc_auc_score,
)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.linear_model import LogisticRegression

from skimage import color, exposure, filters, feature, measure, morphology, segmentation, transform, util
from scipy import ndimage as ndi

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms

import timm
import shap
from captum.attr import Saliency, IntegratedGradients, LayerGradCam, Occlusion

warnings.filterwarnings("ignore")

# ── Reproducibility ────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
FAST_RUN = True   # Set to False for full training (~800 patches/class)

# ── Device ────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ── Class names ───────────────────────────────────────────
CLASS_NAMES = ["IDC_Negative", "IDC_Positive"]
NUM_CLASSES  = len(CLASS_NAMES)

# ── Global CONFIG ─────────────────────────────────────────
CONFIG = {
    "image_size":    224,
    "patch_size":    256,
    "batch_size":    32,
    "num_workers":   2,
    "max_patches_per_class_fast": 100,
    "max_patches_per_class_full": 400,
    "epochs_resnet": 3  if FAST_RUN else 10,
    "epochs_vit":    3  if FAST_RUN else 8,
    "epochs_fusion": 5  if FAST_RUN else 20,
    "lr_resnet":     3e-4,
    "lr_vit":        2e-4,
    "lr_fusion":     1e-3,
    "weight_decay":  1e-4,
    "patience":      5,       # early stopping patience
    "run_lime":      False,
    "num_classes":   NUM_CLASSES,
}

MAX_PER_CLASS = (CONFIG["max_patches_per_class_fast"] if FAST_RUN
                 else CONFIG["max_patches_per_class_full"])

print(f"FAST_RUN={FAST_RUN} | patches per class: {MAX_PER_CLASS} | classes: {CLASS_NAMES}")


## Section 3 — download and load the IDC breast histology dataset
**Source:** Kaggle — "Breast Histopathology Images" by Paul Mooney (CC0 licence).
277 524 50×50 px H&E patches from 162 TCGA-BRCA whole-slide images.
Each patch is labelled 0 (no IDC) or 1 (IDC present).

**Folder structure after download:**
```
tcga_brca_data/IDC_regular_ps50_idx5/<patient_id>/0/  ← IDC negative
tcga_brca_data/IDC_regular_ps50_idx5/<patient_id>/1/  ← IDC positive
```


In [ ]:
# ============================================================
# 3A) DOWNLOAD FROM KAGGLE
# ============================================================
import os

# Paste your Kaggle credentials here (never share them publicly)
os.environ["KAGGLE_USERNAME"] = "YOUR_KAGGLE_USERNAME"   # ← fill in
os.environ["KAGGLE_KEY"]      = "YOUR_KAGGLE_API_KEY"    # ← fill in

!pip install -q kaggle
!kaggle datasets download -d paultimothymooney/breast-histopathology-images \
    -p tcga_brca_data --unzip


In [ ]:
# ============================================================
# 3B) LOAD IDC PATCHES INTO images[] AND labels[]
# ============================================================
from pathlib import Path

DATA_DIR     = Path("tcga_brca_data")
extract_path = DATA_DIR
all_png      = list(extract_path.rglob("*.png"))
print(f"Total patches found: {len(all_png)}")

images_raw, labels_raw, meta_rows = [], [], []

for lid, class_name in enumerate(CLASS_NAMES):
    # Each patch lives in a folder named "0" or "1"
    patches = [p for p in all_png if p.parent.name == str(lid)]
    print(f"{class_name}: {len(patches)} patches available")
    random.shuffle(patches)
    patches = patches[:MAX_PER_CLASS]

    for p in tqdm(patches, desc=class_name, leave=False):
        try:
            img = np.array(Image.open(p).convert("RGB").resize((256, 256)))
            if img.mean() > 240 or img.mean() < 10:
                continue   # skip blank / near-black tiles
            images_raw.append(img)
            labels_raw.append(lid)
            meta_rows.append({"patient_id": p.parent.parent.name, "label": lid})
        except Exception:
            continue

print(f"\nLoaded: {len(images_raw)} patches")
for lid, name in enumerate(CLASS_NAMES):
    n = sum(1 for l in labels_raw if l == lid)
    print(f"  {name}: {n}")


In [ ]:
# ============================================================
# 3C) BALANCE CLASSES AND FINALISE ARRAYS
# ============================================================

images_raw  = np.array(images_raw,  dtype=np.uint8)
labels_raw  = np.array(labels_raw,  dtype=np.int64)
meta_df_raw = pd.DataFrame(meta_rows).reset_index(drop=True)

balanced_idx = []
for lid in range(NUM_CLASSES):
    idx = np.where(labels_raw == lid)[0]
    np.random.shuffle(idx)
    balanced_idx.extend(idx[:MAX_PER_CLASS].tolist())
balanced_idx = np.array(sorted(balanced_idx), dtype=np.int64)

images      = images_raw[balanced_idx]
labels      = labels_raw[balanced_idx]
meta_df     = meta_df_raw.iloc[balanced_idx].reset_index(drop=True)
class_names = CLASS_NAMES

# No clinical features for this dataset
CLINICAL_FEATURE_COLS = []

print("Final dataset shape:", images.shape)
print("Labels:", dict(zip(*np.unique(labels, return_counts=True))))
print("Class names:", class_names)


## Section 4 — basic exploration

In [ ]:
# ============================================================
# 4) BASIC EXPLORATION
# ============================================================

label_counts = pd.Series(labels).value_counts().sort_index()
label_df = pd.DataFrame({
    "class_id":   list(range(len(class_names))),
    "class_name": class_names,
    "count":      label_counts.reindex(range(len(class_names))).values,
})
display(label_df)

plt.figure(figsize=(8, 4))
plt.bar(label_df["class_name"], label_df["count"],
        color=["#2196F3", "#FF5722"])
plt.title("IDC patch class distribution")
plt.ylabel("Number of tiles")
plt.tight_layout(); plt.show()

def show_class_examples(images, labels, class_names, n_per_class=4):
    n_classes = len(class_names)
    fig, axes = plt.subplots(n_classes, n_per_class,
                             figsize=(3 * n_per_class, 3 * n_classes))
    if n_classes == 1:
        axes = np.array([axes])
    for class_id in range(n_classes):
        idx = np.where(labels == class_id)[0]
        chosen = np.random.choice(idx, size=min(n_per_class, len(idx)), replace=False)
        for j in range(n_per_class):
            ax = axes[class_id, j]
            ax.axis("off")
            if j < len(chosen):
                ax.imshow(images[chosen[j]])
                if j == 0:
                    ax.set_title(class_names[class_id], fontsize=9)
    plt.suptitle("Sample H&E tiles — IDC Breast Histopathology", y=1.01, fontsize=13)
    plt.tight_layout(); plt.show()

show_class_examples(images, labels, class_names, n_per_class=4)


## Section 5 — classical segmentation and detection
HED colour deconvolution works on any H&E stained tissue.
`labels_to_boxes` returns boxes with keys `x1, y1, x2, y2` for direct use
in matplotlib `Rectangle` patches.


In [ ]:
# ============================================================
# 5) CLASSICAL SEGMENTATION AND DETECTION
# ============================================================

def normalize_01(x):
    x = x.astype(np.float32)
    x -= x.min(); x /= (x.max() + 1e-8)
    return x

def segment_nuclei_from_histology(img_rgb, min_size=16):
    hed        = color.rgb2hed(img_rgb)
    hematoxylin = normalize_01(-hed[:, :, 0])
    smooth      = filters.gaussian(hematoxylin, sigma=1.0)
    thresh      = filters.threshold_otsu(smooth)
    mask        = smooth > thresh
    mask = morphology.remove_small_objects(mask, min_size=min_size)
    mask = morphology.remove_small_holes(mask, area_threshold=min_size)
    mask = morphology.binary_opening(mask, morphology.disk(1))
    mask = morphology.binary_closing(mask, morphology.disk(1))

    distance = ndi.distance_transform_edt(mask)
    if mask.sum() == 0:
        return hematoxylin, smooth, mask.astype(np.uint8), np.zeros_like(mask, dtype=np.int32)

    coords = feature.peak_local_max(distance, footprint=np.ones((7,7), dtype=bool), labels=mask)
    marker_mask = np.zeros_like(mask, dtype=bool)
    if len(coords):
        marker_mask[tuple(coords.T)] = True
    else:
        marker_mask = mask.copy()

    markers, _ = ndi.label(marker_mask)
    labels_ws   = segmentation.watershed(-distance, markers, mask=mask)
    return hematoxylin, smooth, mask.astype(np.uint8), labels_ws

def labels_to_boxes(labels_ws):
    """Return bounding boxes with keys x1, y1, x2, y2 (image coordinate convention)."""
    boxes = []
    for region in measure.regionprops(labels_ws):
        minr, minc, maxr, maxc = region.bbox
        boxes.append({
            "x1": minc, "y1": minr,
            "x2": maxc, "y2": maxr,
            "area": region.area,
            "eccentricity": region.eccentricity,
        })
    return boxes

# Quick visual check on one tile from each class
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(10, 3.5 * NUM_CLASSES))
if NUM_CLASSES == 1:
    axes = np.array([axes])
for lid in range(NUM_CLASSES):
    idx = np.where(labels == lid)[0][0]
    img = images[idx]
    hem, smooth, mask, labels_ws = segment_nuclei_from_histology(img)
    axes[lid, 0].imshow(img);             axes[lid, 0].set_title(f"{CLASS_NAMES[lid]}\nOriginal", fontsize=8)
    axes[lid, 1].imshow(hem, cmap="Blues"); axes[lid, 1].set_title("Hematoxylin", fontsize=8)
    axes[lid, 2].imshow(mask, cmap="gray"); axes[lid, 2].set_title("Nucleus mask", fontsize=8)
    for ax in axes[lid]: ax.axis("off")
plt.tight_layout(); plt.show()


## Section 6 — handcrafted tabular feature extraction

In [ ]:
# ============================================================
# 6) HANDCRAFTED TABULAR FEATURE EXTRACTION
# ============================================================

def safe_stats(values):
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0:
        return {"mean":0.,"std":0.,"min":0.,"max":0.,"median":0.}
    return {"mean":float(np.mean(values)),"std":float(np.std(values)),
            "min":float(np.min(values)),"max":float(np.max(values)),
            "median":float(np.median(values))}

def glcm_features(gray_u8, distances=(1,2),
                  angles=(0, np.pi/4, np.pi/2, 3*np.pi/4), levels=32):
    gray_small = transform.resize(gray_u8,(64,64),preserve_range=True,anti_aliasing=True).astype(np.uint8)
    quantized  = np.clip(np.floor(gray_small.astype(np.float32)/(256/levels)).astype(np.uint8), 0, levels-1)
    glcm = feature.graycomatrix(quantized, list(distances), list(angles),
                                levels=levels, symmetric=True, normed=True)
    return {f"glcm_{p}": float(feature.graycoprops(glcm, p).mean())
            for p in ["contrast","dissimilarity","homogeneity","energy","correlation","ASM"]}

def lbp_histogram(gray_u8, points=8, radius=1):
    gray_small = transform.resize(gray_u8,(64,64),preserve_range=True,anti_aliasing=True).astype(np.uint8)
    lbp  = feature.local_binary_pattern(gray_small, P=points, R=radius, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=int(points+2), range=(0,int(points+2)), density=True)
    return {f"lbp_{i}": float(hist[i]) for i in range(len(hist))}

def extract_features_from_image(img_rgb):
    feats = {}
    # Colour statistics (RGB + HED)
    for ch, name in enumerate(["R","G","B"]):
        feats.update({f"rgb_{name}_{k}": v
                      for k, v in safe_stats(img_rgb[:,:,ch].astype(np.float32).ravel()).items()})
    hed = color.rgb2hed(img_rgb)
    for ch, name in enumerate(["H","E","D"]):
        feats.update({f"hed_{name}_{k}": v
                      for k, v in safe_stats(hed[:,:,ch].ravel()).items()})
    # Texture
    gray_u8 = (color.rgb2gray(img_rgb) * 255).astype(np.uint8)
    feats.update(glcm_features(gray_u8))
    feats.update(lbp_histogram(gray_u8))
    # Nucleus morphology
    hem, smooth, mask, labels_ws = segment_nuclei_from_histology(img_rgb)
    regions = measure.regionprops(labels_ws)
    boxes   = labels_to_boxes(labels_ws)
    feats["nuclei_count"]       = len(regions)
    feats["nuclei_pixel_ratio"] = float(mask.sum()) / float(mask.size)
    for prop in ["area","eccentricity","perimeter","solidity"]:
        vals = [getattr(r, prop) for r in regions if hasattr(r, prop)]
        feats.update({f"nucleus_{prop}_{k}": v
                      for k, v in safe_stats(vals).items()})
    box_areas = [(b["x2"]-b["x1"])*(b["y2"]-b["y1"]) for b in boxes]
    feats["box_count"] = len(boxes)
    feats.update({f"box_area_{k}": v for k, v in safe_stats(box_areas).items()})
    return feats

print(f"Extracting features from {len(images)} tiles …")
all_feats = []
for i in tqdm(range(len(images))):
    f = extract_features_from_image(images[i])
    f["index"] = i
    f["label"] = int(labels[i])
    all_feats.append(f)

feature_df = pd.DataFrame(all_feats)

# Append clinical features (empty for this dataset)
for col in CLINICAL_FEATURE_COLS:
    if col in meta_df.columns:
        feature_df[col] = meta_df[col].values

feature_df = feature_df.fillna(feature_df.median(numeric_only=True))
print("Feature table shape:", feature_df.shape)
display(feature_df.head(3))


## Section 7 — inspect the tabular modality

In [ ]:
# ============================================================
# 7) INSPECT THE TABULAR MODALITY
# ============================================================

feature_columns = [c for c in feature_df.columns if c not in ["index","label"]]
display(feature_df[feature_columns].describe().T.head(20))

plot_cols = [c for c in ["nuclei_count","nuclei_pixel_ratio",
                          "nucleus_area_mean","glcm_contrast","nucleus_eccentricity_mean"]
             if c in feature_df.columns]

if plot_cols:
    fig, axes = plt.subplots(1, len(plot_cols), figsize=(5*len(plot_cols), 4))
    if len(plot_cols) == 1: axes = [axes]
    for ax, col in zip(axes, plot_cols):
        grouped = [feature_df.loc[feature_df["label"]==lid, col].dropna().values
                   for lid in range(NUM_CLASSES)]
        ax.boxplot(grouped, labels=[n[:8] for n in class_names], showfliers=False)
        ax.set_title(col, fontsize=9)
        ax.tick_params(axis="x", rotation=30)
    plt.suptitle("Feature distributions by class", fontsize=12)
    plt.tight_layout(); plt.show()


## Section 8 — train, validation, and test splits

In [ ]:
# ============================================================
# 8) TRAIN / VALIDATION / TEST SPLITS
# ============================================================

all_indices = np.arange(len(images))
train_val_idx, test_idx, y_train_val, y_test = train_test_split(
    all_indices, labels, test_size=0.20, random_state=SEED, stratify=labels)
train_idx, val_idx, y_train, y_val = train_test_split(
    train_val_idx, y_train_val, test_size=0.20, random_state=SEED, stratify=y_train_val)

print("Train:", len(train_idx), "| Val:", len(val_idx), "| Test:", len(test_idx))

feature_columns = [c for c in feature_df.columns if c not in ["index","label"]]
X_tab = feature_df[feature_columns].values.astype(np.float32)

scaler = StandardScaler().fit(X_tab[train_idx])
X_tab_train = scaler.transform(X_tab[train_idx])
X_tab_val   = scaler.transform(X_tab[val_idx])
X_tab_test  = scaler.transform(X_tab[test_idx])

print("Tabular feature matrix shape:", X_tab.shape)
print("Feature columns sample:", feature_columns[:5], "…")


## Section 8B — feature subset: top-10 correlation-filtered features

In [ ]:
# ============================================================
# 8B) FEATURE SUBSET: TOP 10 CORRELATION-FILTERED FEATURES
# ============================================================

selector_model = RandomForestClassifier(
    n_estimators=300, random_state=SEED, n_jobs=-1, class_weight="balanced_subsample")
selector_model.fit(X_tab_train, y_train)

importances   = pd.Series(selector_model.feature_importances_, index=feature_columns)
top_candidates = importances.sort_values(ascending=False).head(30).index.tolist()

# Use DataFrame indexing (X_tab is a numpy array — must convert first)
X_tab_df   = pd.DataFrame(X_tab, columns=feature_columns)
corr_matrix = X_tab_df[top_candidates].corr().abs()

selected = []
for feat in top_candidates:
    if len(selected) >= 10:
        break
    too_correlated = any(corr_matrix.loc[feat, already] > 0.85 for already in selected)
    if not too_correlated:
        selected.append(feat)

top10_cols = selected
print("Selected top 10 features:", top10_cols)

X_tab_sub       = X_tab_df[top10_cols].copy()
sub_scaler      = StandardScaler()
X_tab_train_sub = sub_scaler.fit_transform(X_tab_sub.iloc[train_idx])
X_tab_val_sub   = sub_scaler.transform(X_tab_sub.iloc[val_idx])
X_tab_test_sub  = sub_scaler.transform(X_tab_sub.iloc[test_idx])

print("Subset train shape:", X_tab_train_sub.shape)


## Section 9 — metrics and calibration helpers

In [ ]:
# ============================================================
# 9) METRICS, CALIBRATION, AND PLOTTING HELPERS
# ============================================================

def one_hot(y, num_classes):
    y = np.asarray(y, dtype=int)
    return np.eye(num_classes, dtype=np.float32)[y]

def expected_calibration_error(probs, y_true, n_bins=15):
    y_true      = np.asarray(y_true)
    probs       = np.asarray(probs)
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies  = (predictions == y_true).astype(np.float32)
    bin_edges   = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        left, right = bin_edges[i], bin_edges[i+1]
        in_bin = ((confidences > left) & (confidences <= right)
                  if i > 0 else (confidences >= left) & (confidences <= right))
        if in_bin.sum() > 0:
            ece += np.abs(accuracies[in_bin].mean() - confidences[in_bin].mean()) * in_bin.mean()
    return float(ece)

def multiclass_brier_score(probs, y_true):
    y_true_oh = one_hot(y_true, probs.shape[1])
    return float(np.mean(np.sum((probs - y_true_oh) ** 2, axis=1)))

def compute_metrics(y_true, probs, model_name="model"):
    preds   = probs.argmax(axis=1)
    metrics = {
        "model":            model_name,
        "accuracy":         accuracy_score(y_true, preds),
        "macro_f1":         f1_score(y_true, preds, average="macro"),
        "balanced_accuracy":balanced_accuracy_score(y_true, preds),
        "log_loss":         log_loss(y_true, probs, labels=list(range(probs.shape[1]))),
        "ece":              expected_calibration_error(probs, y_true),
        "brier":            multiclass_brier_score(probs, y_true),
    }
    try:
        metrics["macro_ovr_auc"] = roc_auc_score(
            one_hot(y_true, probs.shape[1]), probs, average="macro", multi_class="ovr")
    except Exception:
        metrics["macro_ovr_auc"] = np.nan
    return metrics

def print_report(y_true, probs, class_names, title=""):
    preds = probs.argmax(axis=1)
    print(title)
    print(classification_report(y_true, preds, target_names=class_names, digits=4))

def plot_confusion(y_true, probs, class_names, title="Confusion matrix"):
    preds = probs.argmax(axis=1)
    cm    = confusion_matrix(y_true, preds)
    fig, ax = plt.subplots(figsize=(8, 8))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(
        ax=ax, xticks_rotation=45, colorbar=False)
    plt.title(title); plt.show()

def plot_reliability_diagram(probs, y_true, title="Reliability diagram", n_bins=15):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies  = (predictions == y_true).astype(np.float32)
    bin_edges   = np.linspace(0.0, 1.0, n_bins + 1)
    xs, ys, counts = [], [], []
    for i in range(n_bins):
        left, right = bin_edges[i], bin_edges[i+1]
        in_bin = ((confidences > left) & (confidences <= right)
                  if i > 0 else (confidences >= left) & (confidences <= right))
        if in_bin.sum() > 0:
            xs.append(confidences[in_bin].mean())
            ys.append(accuracies[in_bin].mean())
            counts.append(in_bin.sum())
    plt.figure(figsize=(6, 6))
    plt.plot([0,1],[0,1], linestyle="--")
    if xs:
        plt.plot(xs, ys, marker="o")
        for x, y, c in zip(xs, ys, counts):
            plt.text(x, y, str(c), fontsize=8)
    plt.xlabel("Mean confidence"); plt.ylabel("Observed accuracy")
    plt.title(title); plt.grid(alpha=0.3); plt.show()

results = []


## Section 8C — evaluate tabular models on feature subset

In [ ]:
# ============================================================
# 8C) EVALUATE TABULAR MODELS ON THE FEATURE SUBSET
# ============================================================

rf_sub_base  = RandomForestClassifier(
    n_estimators=400 if not FAST_RUN else 150,
    random_state=SEED, n_jobs=-1, class_weight="balanced_subsample")
rf_sub_model = CalibratedClassifierCV(estimator=rf_sub_base, cv=3, method="sigmoid")
rf_sub_model.fit(X_tab_train_sub, y_train)
rf_sub_probs   = rf_sub_model.predict_proba(X_tab_test_sub)
rf_sub_metrics = compute_metrics(y_test, rf_sub_probs, model_name="Tabular RF (top10 subset)")
results.append(rf_sub_metrics)

logreg_sub_model = LogisticRegression(max_iter=2000, random_state=SEED)
logreg_sub_model.fit(X_tab_train_sub, y_train)
logreg_sub_probs   = logreg_sub_model.predict_proba(X_tab_test_sub)
logreg_sub_metrics = compute_metrics(y_test, logreg_sub_probs, model_name="Tabular LR (top10 subset)")
results.append(logreg_sub_metrics)

display(pd.DataFrame([rf_sub_metrics, logreg_sub_metrics]))


## Section 10 — classical AI baselines

In [ ]:
# ============================================================
# 10) CLASSICAL AI BASELINES
# ============================================================
# 10A. TABULAR RANDOM FOREST
# 10B. IMAGE HOG + LINEAR SVM
# Both are classical AI baselines.
# After that we calibrate them and compare the probabilities.

rf_base = RandomForestClassifier( # Builds a random forest on the tabular features as the first baseline
    n_estimators=400 if not FAST_RUN else 150,
    random_state=SEED,
    n_jobs=-1,
    class_weight="balanced_subsample",
)

rf_model = CalibratedClassifierCV(
    estimator=rf_base,
    cv=3,
    method="sigmoid",
)

rf_model.fit(X_tab_train, y_train)

rf_val_probs = rf_model.predict_proba(X_tab_val)
rf_test_probs = rf_model.predict_proba(X_tab_test)

rf_metrics = compute_metrics(y_test, rf_test_probs, model_name="Tabular RandomForest (calibrated)")
results.append(rf_metrics)
display(pd.DataFrame([rf_metrics]))
print_report(y_test, rf_test_probs, class_names, title="Tabular RandomForest report")
plot_confusion(y_test, rf_test_probs, class_names, title="Tabular RandomForest confusion matrix")
plot_reliability_diagram(rf_test_probs, y_test, title="Tabular RandomForest reliability")

def extract_hog_vector(img_rgb):
    gray = color.rgb2gray(transform.resize(img_rgb, (128, 128), preserve_range=True, anti_aliasing=True))
    hog_vec = feature.hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8), # Each 8x8 pixel patch gets one gradient histogram
        cells_per_block=(2, 2), # Groups 2x2 cells together for local normalization
        block_norm="L2-Hys",
        feature_vector=True,
    )
    return hog_vec.astype(np.float32)

HOG_CACHE_NPZ = "crc_hog_features.npz"

if os.path.exists(HOG_CACHE_NPZ):
    print("Loading cached HOG features...")
    hog_data = np.load(HOG_CACHE_NPZ)
    X_hog = hog_data["X_hog"]
else:
    hog_vectors = []
    for idx in tqdm(range(len(images)), desc="Extracting HOG features"):
        hog_vectors.append(extract_hog_vector(images[idx]))
    X_hog = np.stack(hog_vectors)
    np.savez_compressed(HOG_CACHE_NPZ, X_hog=X_hog)

X_hog_train = X_hog[train_idx]
X_hog_val = X_hog[val_idx]
X_hog_test = X_hog[test_idx]

hog_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", CalibratedClassifierCV(
        estimator=LinearSVC(class_weight="balanced", random_state=SEED),
        cv=3,
        method="sigmoid",
    )),
])

hog_svm.fit(X_hog_train, y_train)

hog_val_probs = hog_svm.predict_proba(X_hog_val)
hog_test_probs = hog_svm.predict_proba(X_hog_test)

hog_metrics = compute_metrics(y_test, hog_test_probs, model_name="Image HOG + LinearSVM (calibrated)")
results.append(hog_metrics)
display(pd.DataFrame([hog_metrics]))
print_report(y_test, hog_test_probs, class_names, title="HOG + LinearSVM report")
plot_confusion(y_test, hog_test_probs, class_names, title="HOG + LinearSVM confusion matrix")
plot_reliability_diagram(hog_test_probs, y_test, title="HOG + LinearSVM reliability")

## Section 10C — additional tabular model: logistic regression

In [ ]:
# ============================================================
# 10C) ADDITIONAL TABULAR MODEL: LOGISTIC REGRESSION
# ============================================================

logreg_model = LogisticRegression(
    max_iter=2000,
    random_state=SEED,
)

logreg_model.fit(X_tab_train, y_train)

logreg_val_probs = logreg_model.predict_proba(X_tab_val)
logreg_test_probs = logreg_model.predict_proba(X_tab_test)

logreg_metrics = compute_metrics(
    y_test,
    logreg_test_probs,
    model_name="Tabular LogisticRegression"
)
results.append(logreg_metrics)
display(pd.DataFrame([logreg_metrics]))
print_report(y_test, logreg_test_probs, class_names, title="Tabular LogisticRegression report")
plot_confusion(y_test, logreg_test_probs, class_names, title="Tabular LogisticRegression confusion matrix")
plot_reliability_diagram(logreg_test_probs, y_test, title="Tabular LogisticRegression reliability")

## Section 11 — PyTorch datasets, transforms, and image models

In [ ]:
# ============================================================
# 11) PYTORCH DATASETS, TRANSFORMS, AND IMAGE MODELS
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])), #Resizes all images to 224x224 so they match the pretrained model's expected input
    transforms.RandomHorizontalFlip(), # Randomly flips images left-right to help the model generalize
    transforms.RandomVerticalFlip(), #  Randomly flips images top-bottom
    transforms.RandomRotation(10), # Small random rotations to add more variety during training
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), # Scales pixel values to match what the pretrained ImageNet models expect
])

eval_tfms = transforms.Compose([
    transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class HistologyImageDataset(Dataset):
    def __init__(self, images, labels, indices, transform=None):
        self.images = images
        self.labels = labels
        self.indices = np.array(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        img = Image.fromarray(self.images[idx])
        if self.transform is not None:
            img = self.transform(img)
        label = int(self.labels[idx])
        return {
            "image": img,
            "label": torch.tensor(label, dtype=torch.long),
            "index": torch.tensor(idx, dtype=torch.long),
        }

train_ds_img = HistologyImageDataset(images, labels, train_idx, transform=train_tfms)
val_ds_img = HistologyImageDataset(images, labels, val_idx, transform=eval_tfms)
test_ds_img = HistologyImageDataset(images, labels, test_idx, transform=eval_tfms)

train_loader_img = DataLoader(
    train_ds_img,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)
val_loader_img = DataLoader(
    val_ds_img,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)
test_loader_img = DataLoader(
    test_ds_img,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)

class TimmImageClassifier(nn.Module):
    def __init__(self, model_name, num_classes, pretrained=True, dropout=0.2):
        super().__init__()
        self.model_name = model_name
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool="")
        self.num_features = self.backbone.num_features
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(self.num_features, num_classes)

    def extract_features(self, x):
        feats = self.backbone.forward_features(x)
        if feats.ndim == 4:
            feats = F.adaptive_avg_pool2d(feats, output_size=1).flatten(1)
        elif feats.ndim == 3:
            feats = feats[:, 0]
        return feats

    def forward(self, x):
        feats = self.extract_features(x)
        logits = self.head(self.dropout(feats))
        return logits

def freeze_for_fast_finetuning(model):
    # Freeze everything first
    for p in model.parameters():
        p.requires_grad = False

    # Always train the classification head
    for p in model.head.parameters():
        p.requires_grad = True

    # Unfreeze a small number of late layers depending on architecture
    if hasattr(model.backbone, "layer4"):  # ResNet-like
        for p in model.backbone.layer4.parameters():
            p.requires_grad = True
    elif hasattr(model.backbone, "blocks"):  # ViT-like
        for block in model.backbone.blocks[-2:]:
            for p in block.parameters():
                p.requires_grad = True
        if hasattr(model.backbone, "norm"):
            for p in model.backbone.norm.parameters():
                p.requires_grad = True

num_classes = len(class_names)

resnet_model = TimmImageClassifier("resnet18", num_classes=num_classes, pretrained=True)
vit_model = TimmImageClassifier("vit_tiny_patch16_224", num_classes=num_classes, pretrained=True)

freeze_for_fast_finetuning(resnet_model)
freeze_for_fast_finetuning(vit_model)

resnet_model = resnet_model.to(DEVICE)
vit_model = vit_model.to(DEVICE)

trainable_resnet = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
trainable_vit = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)

print("Trainable parameters - ResNet18:", f"{trainable_resnet:,}")
print("Trainable parameters - ViT tiny:", f"{trainable_vit:,}")

## Section 12 — deep model training utilities

In [ ]:
# ============================================================
# 12) TRAINING UTILITIES FOR DEEP MODELS
# ============================================================

def build_class_weight_tensor(y_train, num_classes):
    class_counts = np.bincount(y_train, minlength=num_classes).astype(np.float32)
    class_weights = class_counts.sum() / np.maximum(class_counts, 1.0)
    class_weights = class_weights / class_weights.mean()
    return torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

CLASS_WEIGHTS = build_class_weight_tensor(y_train, num_classes)

def collect_logits_and_labels(model, loader):
    model.eval()
    all_logits = []
    all_labels = []
    all_indices = []
    with torch.no_grad():
        for batch in loader:
            x = batch["image"].to(DEVICE, non_blocking=True)
            y = batch["label"].to(DEVICE, non_blocking=True)
            logits = model(x)
            all_logits.append(logits.detach().cpu())
            all_labels.append(y.detach().cpu())
            all_indices.append(batch["index"].detach().cpu())
    logits = torch.cat(all_logits).numpy()
    labels_np = torch.cat(all_labels).numpy()
    indices_np = torch.cat(all_indices).numpy()
    return logits, labels_np, indices_np

def logits_to_probs(logits):
    logits_t = torch.tensor(logits, dtype=torch.float32)
    return F.softmax(logits_t, dim=1).numpy()

def fit_temperature_scaler(logits, labels, max_iter=100):
    logits_t = torch.tensor(logits, dtype=torch.float32, device=DEVICE)
    labels_t = torch.tensor(labels, dtype=torch.long, device=DEVICE)
    temperature = nn.Parameter(torch.ones(1, device=DEVICE))

    optimizer = torch.optim.LBFGS([temperature], lr=0.01, max_iter=max_iter)
    criterion = nn.CrossEntropyLoss()

    def closure():
        optimizer.zero_grad()
        loss = criterion(logits_t / temperature.clamp(min=1e-3), labels_t)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(temperature.detach().cpu().item())

def apply_temperature(logits, temperature):
    logits_t = torch.tensor(logits, dtype=torch.float32)
    probs = F.softmax(logits_t / float(temperature), dim=1).numpy()
    return probs

def train_image_model(model, train_loader, val_loader, epochs, lr, weight_decay, model_name="model"):
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    history = []
    best_state = deepcopy(model.state_dict())
    best_f1 = -np.inf
    patience_counter = 0 # Tracks how many epochs in a row validation didn't improve

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        train_labels = []
        train_probs = []

        for batch in tqdm(train_loader, desc=f"{model_name} | epoch {epoch}/{epochs}", leave=False):
            x = batch["image"].to(DEVICE, non_blocking=True)
            y = batch["label"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(x)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * x.size(0)
            train_labels.append(y.detach().cpu())
            train_probs.append(F.softmax(logits.detach().cpu(), dim=1))

        scheduler.step()

        train_labels_np = torch.cat(train_labels).numpy()
        train_probs_np = torch.cat(train_probs).numpy()
        train_metrics = compute_metrics(train_labels_np, train_probs_np, model_name=f"{model_name}_train")

        val_logits, val_labels_np, _ = collect_logits_and_labels(model, val_loader)
        val_probs_np = logits_to_probs(val_logits)
        val_metrics = compute_metrics(val_labels_np, val_probs_np, model_name=f"{model_name}_val")

        epoch_record = { # Saves train and validation scores for this epoch so we can plot them later
            "epoch": epoch,
            "train_loss": running_loss / len(train_loader.dataset),
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_log_loss": val_metrics["log_loss"],
            "val_ece": val_metrics["ece"],
        }
        history.append(epoch_record)

        print(
            f'{model_name} | epoch {epoch}/{epochs} | '
            f'train_loss={epoch_record["train_loss"]:.4f} | '
            f'train_f1={epoch_record["train_macro_f1"]:.4f} | '
            f'val_f1={epoch_record["val_macro_f1"]:.4f} | '
            f'val_acc={epoch_record["val_accuracy"]:.4f}'
        )

        if val_metrics["macro_f1"] > best_f1:
            best_f1 = val_metrics["macro_f1"]
            best_state = deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= CONFIG["patience"]: # If no improvement for several epochs, stop training early
                print(f"Early stopping triggered for {model_name}.") # Tells us training ended early because the model stopped getting better
                break

    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history)
    return model, history_df

def plot_history(history_df, title="Training history"):
    fig, axes = plt.subplots(1, 2, figsize=(12 , 4))

    axes[0].plot(history_df["epoch"], history_df["train_macro_f1"], marker="o", label="train")
    axes[0].plot(history_df["epoch"], history_df["val_macro_f1"], marker="o", label="validation")
    axes[0].set_title(f"{title} - macro F1")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Macro F1")
    axes[0].legend()

    axes[1].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train loss")
    axes[1].plot(history_df["epoch"], history_df["val_log_loss"], marker="o", label="validation log loss")
    axes[1].set_title(f"{title} - loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


## Section 13 — train the ResNet18 image model

In [ ]:
# ============================================================
# 13) TRAIN THE RESNET18 IMAGE MODEL
# ============================================================

resnet_model, resnet_history = train_image_model(
    resnet_model,
    train_loader_img,
    val_loader_img,
    epochs=CONFIG["epochs_resnet"],
    lr=CONFIG["lr_resnet"],
    weight_decay=CONFIG["weight_decay"],
    model_name="ResNet18",
)

display(resnet_history)
plot_history(resnet_history, title="ResNet18")

resnet_val_logits, resnet_val_labels, resnet_val_indices = collect_logits_and_labels(resnet_model, val_loader_img)
resnet_test_logits, resnet_test_labels, resnet_test_indices = collect_logits_and_labels(resnet_model, test_loader_img)

resnet_test_probs_raw = logits_to_probs(resnet_test_logits)
resnet_temp = fit_temperature_scaler(resnet_val_logits, resnet_val_labels)
resnet_test_probs_cal = apply_temperature(resnet_test_logits, resnet_temp)

resnet_raw_metrics = compute_metrics(resnet_test_labels, resnet_test_probs_raw, model_name="ResNet18 raw")
resnet_cal_metrics = compute_metrics(resnet_test_labels, resnet_test_probs_cal, model_name="ResNet18 calibrated")

results.append(resnet_raw_metrics)
results.append(resnet_cal_metrics)

display(pd.DataFrame([resnet_raw_metrics, resnet_cal_metrics]))
print("Best temperature for ResNet18:", resnet_temp)

print_report(resnet_test_labels, resnet_test_probs_cal, class_names, title="ResNet18 calibrated report")
plot_confusion(resnet_test_labels, resnet_test_probs_cal, class_names, title="ResNet18 calibrated confusion matrix")
plot_reliability_diagram(resnet_test_probs_raw, resnet_test_labels, title="ResNet18 reliability before calibration")
plot_reliability_diagram(resnet_test_probs_cal, resnet_test_labels, title="ResNet18 reliability after calibration")

## Section 14 — train the ViT image model

In [ ]:
# ============================================================
# 14) TRAIN THE ViT IMAGE MODEL
# ============================================================

vit_model, vit_history = train_image_model(
    vit_model,
    train_loader_img,
    val_loader_img,
    epochs=CONFIG["epochs_vit"],
    lr=CONFIG["lr_vit"],
    weight_decay=CONFIG["weight_decay"],
    model_name="ViT-tiny",
)

display(vit_history)
plot_history(vit_history, title="ViT-tiny")

vit_val_logits, vit_val_labels, vit_val_indices = collect_logits_and_labels(vit_model, val_loader_img)
vit_test_logits, vit_test_labels, vit_test_indices = collect_logits_and_labels(vit_model, test_loader_img)

vit_test_probs_raw = logits_to_probs(vit_test_logits)
vit_temp = fit_temperature_scaler(vit_val_logits, vit_val_labels)
vit_test_probs_cal = apply_temperature(vit_test_logits, vit_temp)

vit_raw_metrics = compute_metrics(vit_test_labels, vit_test_probs_raw, model_name="ViT-tiny raw")
vit_cal_metrics = compute_metrics(vit_test_labels, vit_test_probs_cal, model_name="ViT-tiny calibrated")

results.append(vit_raw_metrics)
results.append(vit_cal_metrics)

display(pd.DataFrame([vit_raw_metrics, vit_cal_metrics]))
print("Best temperature for ViT-tiny:", vit_temp)

print_report(vit_test_labels, vit_test_probs_cal, class_names, title="ViT-tiny calibrated report")
plot_confusion(vit_test_labels, vit_test_probs_cal, class_names, title="ViT-tiny calibrated confusion matrix")
plot_reliability_diagram(vit_test_probs_raw, vit_test_labels, title="ViT-tiny reliability before calibration")
plot_reliability_diagram(vit_test_probs_cal, vit_test_labels, title="ViT-tiny reliability after calibration")

## Section 14B — additional image model: MobileNetV3-Small

In [ ]:
# ============================================================
# 14B) ADDITIONAL IMAGE MODEL: MOBILENETV3-SMALL
# ============================================================

extra_img_model = TimmImageClassifier(
    "mobilenetv3_small_100",
    num_classes=num_classes,
    pretrained=True
)

freeze_for_fast_finetuning(extra_img_model)
extra_img_model = extra_img_model.to(DEVICE)

trainable_extra = sum(p.numel() for p in extra_img_model.parameters() if p.requires_grad)
print("Trainable parameters - MobileNetV3-Small:", f"{trainable_extra:,}")

extra_img_model, extra_history = train_image_model(
    extra_img_model,
    train_loader_img,
    val_loader_img,
    epochs=CONFIG["epochs_resnet"],
    lr=CONFIG["lr_resnet"],
    weight_decay=CONFIG["weight_decay"],
    model_name="MobileNetV3-Small",
)

display(extra_history)
plot_history(extra_history, title="MobileNetV3-Small")

extra_val_logits, extra_val_labels, extra_val_indices = collect_logits_and_labels(extra_img_model, val_loader_img)
extra_test_logits, extra_test_labels, extra_test_indices = collect_logits_and_labels(extra_img_model, test_loader_img)

extra_test_probs_raw = logits_to_probs(extra_test_logits)
extra_temp = fit_temperature_scaler(extra_val_logits, extra_val_labels)
extra_test_probs_cal = apply_temperature(extra_test_logits, extra_temp)

extra_raw_metrics = compute_metrics(extra_test_labels, extra_test_probs_raw, model_name="MobileNetV3-Small raw")
extra_cal_metrics = compute_metrics(extra_test_labels, extra_test_probs_cal, model_name="MobileNetV3-Small calibrated")

results.append(extra_raw_metrics)
results.append(extra_cal_metrics)

display(pd.DataFrame([extra_raw_metrics, extra_cal_metrics]))
print("Best temperature for MobileNetV3-Small:", extra_temp)

print_report(extra_test_labels, extra_test_probs_cal, class_names, title="MobileNetV3-Small calibrated report")

## Section 15 — extract image embeddings

In [ ]:
# ============================================================
# 15) EXTRACT IMAGE EMBEDDINGS FOR MULTIMODAL FUSION
# ============================================================

def collect_embeddings_logits_labels(model, loader):
    model.eval()
    emb_dict = {}
    logit_dict = {}
    label_dict = {}
    with torch.no_grad():
        for batch in tqdm(loader, desc="Extracting embeddings", leave=False):
            x = batch["image"].to(DEVICE, non_blocking=True)
            y = batch["label"].cpu().numpy()
            idxs = batch["index"].cpu().numpy()

            feats = model.extract_features(x).detach().cpu().numpy()
            logits = model(x).detach().cpu().numpy()

            for idx, feat, logit, label in zip(idxs, feats, logits, y):
                emb_dict[int(idx)] = feat.astype(np.float32)
                logit_dict[int(idx)] = logit.astype(np.float32)
                label_dict[int(idx)] = int(label)

    return emb_dict, logit_dict, label_dict

full_eval_ds = HistologyImageDataset(images, labels, np.arange(len(images)), transform=eval_tfms)
full_eval_loader = DataLoader(
    full_eval_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)

vit_emb_dict, vit_logit_dict, label_dict = collect_embeddings_logits_labels(vit_model, full_eval_loader)
resnet_emb_dict, resnet_logit_dict, _ = collect_embeddings_logits_labels(resnet_model, full_eval_loader)

def build_array_from_dict(feature_dict, indices):
    return np.stack([feature_dict[int(i)] for i in indices])

vit_emb_train = build_array_from_dict(vit_emb_dict, train_idx)
vit_emb_val = build_array_from_dict(vit_emb_dict, val_idx)
vit_emb_test = build_array_from_dict(vit_emb_dict, test_idx)

resnet_emb_train = build_array_from_dict(resnet_emb_dict, train_idx)
resnet_emb_val = build_array_from_dict(resnet_emb_dict, val_idx)
resnet_emb_test = build_array_from_dict(resnet_emb_dict, test_idx)

print("ViT embedding shape:", vit_emb_train.shape)
print("ResNet embedding shape:", resnet_emb_train.shape)
print("Tabular feature shape:", X_tab_train.shape)

## Section 16 — fusion dataset and fusion models

In [ ]:
# ============================================================
# 16) FUSION DATASET AND FUSION MODELS
# ============================================================
# We implement the major practical fusion families:
# - early / feature-level fusion: concatenate image and tabular vectors,
# - gated intermediate fusion: learn how much to trust each modality,
# - cross-attention fusion: let the modalities interact through attention,
# - tensor fusion: multiplicative interactions through an outer product,
# - late fusion: combine probabilities after separate models decide.

class EmbeddingFusionDataset(Dataset): # Pairs image embeddings with tabular features and labels for the fusion models
    def __init__(self, img_features, tab_features, labels):
        self.img_features = np.asarray(img_features, dtype=np.float32)
        self.tab_features = np.asarray(tab_features, dtype=np.float32)
        self.labels = np.asarray(labels, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {
            "img_emb": torch.tensor(self.img_features[i], dtype=torch.float32),
            "tab": torch.tensor(self.tab_features[i], dtype=torch.float32),
            "label": torch.tensor(self.labels[i], dtype=torch.long),
        }

train_ds_fusion = EmbeddingFusionDataset(vit_emb_train, X_tab_train, y_train)
val_ds_fusion = EmbeddingFusionDataset(vit_emb_val, X_tab_val, y_val)
test_ds_fusion = EmbeddingFusionDataset(vit_emb_test, X_tab_test, y_test)

train_loader_fusion = DataLoader(train_ds_fusion, batch_size=64, shuffle=True, num_workers=0)
val_loader_fusion = DataLoader(val_ds_fusion, batch_size=128, shuffle=False, num_workers=0)
test_loader_fusion = DataLoader(test_ds_fusion, batch_size=128, shuffle=False, num_workers=0)

class EarlyConcatFusionNet(nn.Module): # Joins image and tabular vectors into one big vector before classifying
    def __init__(self, img_dim, tab_dim, num_classes, hidden=256, dropout=0.3):
        super().__init__()
        self.img_proj = nn.Sequential( # Projects image embeddings down to a fixed hidden size
            nn.Linear(img_dim, hidden), # Linear layer that maps image embedding to hidden dimensions
            nn.BatchNorm1d(hidden), # Normalizes values across the batch for stable training
            nn.ReLU(), # Activation function that keeps positive values and zeros out negatives
            nn.Dropout(dropout), # Randomly drops some values during training to prevent overfitting
        )
        self.tab_proj = nn.Sequential( # Same projection but for the tabular features
            nn.Linear(tab_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Sequential( # Takes the combined vector and outputs class scores
            nn.Linear(hidden * 2, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, img_feat, tab_feat, return_parts=False): # Runs one forward pass: project, combine, classify
        img_z = self.img_proj(img_feat) # Project image embedding
        tab_z = self.tab_proj(tab_feat) # Project tabular features
        fused = torch.cat([img_z, tab_z], dim=1) # Concatenate both projections into one vector
        logits = self.classifier(fused) # Feed the combined vector through the classifier to get predictions
        if return_parts:
            return logits, {"img_proj": img_z, "tab_proj": tab_z}
        return logits

class GatedFusionNet(nn.Module):
    def __init__(self, img_dim, tab_dim, num_classes, hidden=256, dropout=0.3):
        super().__init__()
        self.img_proj = nn.Sequential(
            nn.Linear(img_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.tab_proj = nn.Sequential(
            nn.Linear(tab_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.gate = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 2),
            nn.Softmax(dim=1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, img_feat, tab_feat, return_parts=False):
        img_z = self.img_proj(img_feat)
        tab_z = self.tab_proj(tab_feat)
        gates = self.gate(torch.cat([img_z, tab_z], dim=1))
        fused = gates[:, 0:1] * img_z + gates[:, 1:2] * tab_z
        logits = self.classifier(fused)
        if return_parts:
            return logits, {"gates": gates, "img_proj": img_z, "tab_proj": tab_z}
        return logits

class CrossAttentionFusionNet(nn.Module):
    def __init__(self, img_dim, tab_dim, num_classes, hidden=256, dropout=0.3, num_heads=4):
        super().__init__()
        self.img_proj = nn.Linear(img_dim, hidden)
        self.tab_proj = nn.Linear(tab_dim, hidden)
        self.attn = nn.MultiheadAttention(embed_dim=hidden, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(hidden)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, img_feat, tab_feat, return_parts=False):
        img_tok = self.img_proj(img_feat)
        tab_tok = self.tab_proj(tab_feat)
        tokens = torch.stack([img_tok, tab_tok], dim=1)  # [B, 2, H]
        attn_out, attn_weights = self.attn(
            tokens,
            tokens,
            tokens,
            need_weights=True,
            average_attn_weights=False,
        )
        fused = self.norm(attn_out.mean(dim=1))
        logits = self.classifier(fused)
        if return_parts:
            return logits, {"attention_weights": attn_weights, "tokens": tokens, "attn_out": attn_out}
        return logits

class TensorFusionNet(nn.Module):
    def __init__(self, img_dim, tab_dim, num_classes, hidden=64, dropout=0.3):
        super().__init__()
        self.img_proj = nn.Sequential(
            nn.Linear(img_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.tab_proj = nn.Sequential(
            nn.Linear(tab_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        fusion_dim = (hidden + 1) * (hidden + 1)
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, img_feat, tab_feat, return_parts=False):
        img_z = self.img_proj(img_feat)
        tab_z = self.tab_proj(tab_feat)

        ones_img = torch.ones(img_z.size(0), 1, device=img_z.device)
        ones_tab = torch.ones(tab_z.size(0), 1, device=tab_z.device)

        img_aug = torch.cat([ones_img, img_z], dim=1)
        tab_aug = torch.cat([ones_tab, tab_z], dim=1)

        outer = torch.bmm(img_aug.unsqueeze(2), tab_aug.unsqueeze(1)).flatten(1)
        logits = self.classifier(outer)

        if return_parts:
            return logits, {"img_proj": img_z, "tab_proj": tab_z, "outer": outer}
        return logits

img_dim = vit_emb_train.shape[1]
tab_dim = X_tab_train.shape[1]

fusion_models = { # Dictionary holding all four fusion model architectures ready to train
    "EarlyConcatFusion": EarlyConcatFusionNet(img_dim, tab_dim, num_classes),
    "GatedFusion": GatedFusionNet(img_dim, tab_dim, num_classes),
    "CrossAttentionFusion": CrossAttentionFusionNet(img_dim, tab_dim, num_classes),
    "TensorFusion": TensorFusionNet(img_dim, tab_dim, num_classes),
}

## Section 17 — fusion training loop

In [ ]:
# ============================================================
# 17) TRAINING LOOP FOR FUSION MODELS
# ============================================================

def collect_fusion_logits_labels(model, loader):
    model.eval()
    all_logits = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            img_feat = batch["img_emb"].to(DEVICE)
            tab_feat = batch["tab"].to(DEVICE)
            y = batch["label"].to(DEVICE)
            logits = model(img_feat, tab_feat)
            all_logits.append(logits.detach().cpu())
            all_labels.append(y.detach().cpu())
    logits = torch.cat(all_logits).numpy()
    labels_np = torch.cat(all_labels).numpy()
    return logits, labels_np

def train_fusion_model(model, train_loader, val_loader, epochs, lr, weight_decay, model_name="fusion_model"): # Trains a fusion model using image embeddings and tabular features together
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    history = []
    best_state = deepcopy(model.state_dict())
    best_f1 = -np.inf
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        train_labels = []
        train_probs = []

        for batch in tqdm(train_loader, desc=f"{model_name} | epoch {epoch}/{epochs}", leave=False):
            img_feat = batch["img_emb"].to(DEVICE)
            tab_feat = batch["tab"].to(DEVICE)
            y = batch["label"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(img_feat, tab_feat)
                loss = criterion(logits, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * y.size(0)
            train_labels.append(y.detach().cpu())
            train_probs.append(F.softmax(logits.detach().cpu(), dim=1))

        scheduler.step()

        train_labels_np = torch.cat(train_labels).numpy()
        train_probs_np = torch.cat(train_probs).numpy()
        train_metrics = compute_metrics(train_labels_np, train_probs_np, model_name=f"{model_name}_train")

        val_logits, val_labels_np = collect_fusion_logits_labels(model, val_loader)
        val_probs_np = logits_to_probs(val_logits)
        val_metrics = compute_metrics(val_labels_np, val_probs_np, model_name=f"{model_name}_val")

        epoch_record = {
            "epoch": epoch,
            "train_loss": running_loss / len(train_loader.dataset),
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_log_loss": val_metrics["log_loss"],
            "val_ece": val_metrics["ece"],
        }
        history.append(epoch_record)

        print(
            f'{model_name} | epoch {epoch}/{epochs} | '
            f'train_loss={epoch_record["train_loss"]:.4f} | '
            f'train_f1={epoch_record["train_macro_f1"]:.4f} | '
            f'val_f1={epoch_record["val_macro_f1"]:.4f} | '
            f'val_acc={epoch_record["val_accuracy"]:.4f}'
        )

        if val_metrics["macro_f1"] > best_f1:
            best_f1 = val_metrics["macro_f1"]
            best_state = deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= CONFIG["patience"]:
                print(f"Early stopping triggered for {model_name}.")
                break

    model.load_state_dict(best_state)
    history_df = pd.DataFrame(history)
    return model, history_df

## Section 18 — train and evaluate all fusion models

In [ ]:
# ============================================================
# 18) TRAIN AND EVALUATE ALL FUSION MODELS
# ============================================================

trained_fusion_models = {}
fusion_histories = {}
fusion_val_logits = {}
fusion_test_logits = {}
fusion_temperatures = {}
fusion_test_probs_cal = {}

for fusion_name, fusion_model in fusion_models.items(): # Loops through each fusion model, trains it, calibrates it, and saves results
    print("=" * 80) # Prints a separator line to keep the output readable between models
    print("Training:", fusion_name)

    trained_model, history_df = train_fusion_model(
        fusion_model,
        train_loader_fusion,
        val_loader_fusion,
        epochs=CONFIG["epochs_fusion"],
        lr=CONFIG["lr_fusion"],
        weight_decay=CONFIG["weight_decay"],
        model_name=fusion_name,
    )

    trained_fusion_models[fusion_name] = trained_model
    fusion_histories[fusion_name] = history_df

    display(history_df.tail())
    plot_history(history_df, title=fusion_name)

    val_logits, val_labels_f = collect_fusion_logits_labels(trained_model, val_loader_fusion)
    test_logits, test_labels_f = collect_fusion_logits_labels(trained_model, test_loader_fusion)

    fusion_val_logits[fusion_name] = val_logits
    fusion_test_logits[fusion_name] = test_logits

    temp = fit_temperature_scaler(val_logits, val_labels_f)
    fusion_temperatures[fusion_name] = temp
    probs_cal = apply_temperature(test_logits, temp)
    fusion_test_probs_cal[fusion_name] = probs_cal

    metrics_row = compute_metrics(test_labels_f, probs_cal, model_name=f"{fusion_name} calibrated")
    results.append(metrics_row)

    display(pd.DataFrame([metrics_row]))
    print("Temperature:", temp)
    plot_reliability_diagram(probs_cal, test_labels_f, title=f"{fusion_name} reliability after calibration")

print("Finished training all fusion models.")

## Section 19 — late fusion and stacking

In [ ]:
# ============================================================
# 19) LATE FUSION AND STACKING
# ============================================================
# Here every model makes its own prediction first.
# Then we combine probabilities at the decision level.

# Build validation probabilities for neural models
resnet_val_probs_raw = logits_to_probs(resnet_val_logits)
resnet_val_probs_cal = apply_temperature(resnet_val_logits, resnet_temp)

vit_val_probs_raw = logits_to_probs(vit_val_logits)
vit_val_probs_cal = apply_temperature(vit_val_logits, vit_temp)

# Calibrated classical validation probabilities already exist:
# rf_val_probs and hog_val_probs

base_val_prob_dict = { # Validation probabilities from each base model, used to find the best fusion weights
    "rf": rf_val_probs,
    "hog": hog_val_probs,
    "resnet": resnet_val_probs_cal,
    "vit": vit_val_probs_cal,
}

base_test_prob_dict = { # Test probabilities from each base model, used to make the final late fusion prediction
    "rf": rf_test_probs,
    "hog": hog_test_probs,
    "resnet": resnet_test_probs_cal,
    "vit": vit_test_probs_cal,
}

def normalize_weight_vector(w):
    w = np.array(w, dtype=np.float64)
    w = np.clip(w, 0.0, None)
    if w.sum() == 0:
        w = np.ones_like(w)
    return w / w.sum()

def weighted_prob_average(prob_list, weights):
    weights = normalize_weight_vector(weights)
    out = np.zeros_like(prob_list[0], dtype=np.float64)
    for p, w in zip(prob_list, weights):
        out += w * p
    out = np.clip(out, 1e-8, 1.0)
    out = out / out.sum(axis=1, keepdims=True)
    return out

candidate_models = ["rf", "hog", "resnet", "vit"]

best_score = np.inf
best_weights = None

# Coarse but practical grid search for four models
grid = np.linspace(0.0, 1.0, 11)
for w1 in grid:
    for w2 in grid:
        for w3 in grid:
            for w4 in grid:
                if (w1 + w2 + w3 + w4) == 0:
                    continue
                weights = normalize_weight_vector([w1, w2, w3, w4])
                val_probs = weighted_prob_average(
                    [base_val_prob_dict[m] for m in candidate_models],
                    weights,
                )
                score = log_loss(y_val, val_probs, labels=list(range(num_classes)))
                if score < best_score:
                    best_score = score
                    best_weights = weights

late_fusion_test_probs = weighted_prob_average(
    [base_test_prob_dict[m] for m in candidate_models],
    best_weights,
)
late_fusion_metrics = compute_metrics(y_test, late_fusion_test_probs, model_name="LateFusion weighted average") # Scores the weighted average late fusion on the test set
results.append(late_fusion_metrics)

print("Best late-fusion weights:")
for name, weight in zip(candidate_models, best_weights):
    print(f"  {name}: {weight:.3f}")
display(pd.DataFrame([late_fusion_metrics]))
plot_reliability_diagram(late_fusion_test_probs, y_test, title="LateFusion reliability") # Checks if the late fusion confidence scores are well calibrated

# Stacking meta-learner
X_meta_val = np.hstack([base_val_prob_dict[m] for m in candidate_models])
X_meta_test = np.hstack([base_test_prob_dict[m] for m in candidate_models])

stacker = LogisticRegression(
    max_iter=2000,
    solver="lbfgs",
    random_state=SEED,
)
stacker.fit(X_meta_val, y_val)

stack_test_probs = stacker.predict_proba(X_meta_test)
stack_metrics = compute_metrics(y_test, stack_test_probs, model_name="Stacking ensemble")
results.append(stack_metrics)

display(pd.DataFrame([stack_metrics]))
plot_reliability_diagram(stack_test_probs, y_test, title="Stacking reliability")

## Section 20 — compare all models

In [ ]:
# ============================================================
# 20) COMPARE ALL MODELS IN ONE TABLE
# ============================================================

results_df = pd.DataFrame(results).sort_values( # Puts all model results into one table sorted by best performance
    by=["macro_f1", "accuracy", "macro_ovr_auc"], # Sorts by F1 first, then accuracy, then AUC as tiebreakers
    ascending=False, # Highest scores on top
).reset_index(drop=True)

display(results_df)

metric_cols = ["accuracy", "macro_f1", "balanced_accuracy", "macro_ovr_auc", "log_loss", "ece", "brier"]
for metric in metric_cols:
    plt.figure(figsize=(10, 4))
    plt.bar(results_df["model"], results_df[metric])
    plt.xticks(rotation=60, ha="right")
    plt.title(f"Model comparison: {metric}")
    plt.tight_layout()
    plt.show()

best_model_name = results_df.iloc[0]["model"]
print("Best overall model in this run:", best_model_name)

## Section 21 — CNN image interpretability (ResNet18)

In [ ]:
# ============================================================
# 21) IMAGE INTERPRETABILITY FOR THE CNN (RESNET18)
# ============================================================
# We use several explanation families:
# - saliency maps,
# - integrated gradients,
# - Grad-CAM,
# - occlusion sensitivity.
#
# This gives both gradient-based and perturbation-based explanations.

def denormalize_tensor(img_tensor): # Reverses the ImageNet normalization so we can display the image normally
    mean = torch.tensor(IMAGENET_MEAN, device=img_tensor.device).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD, device=img_tensor.device).view(3, 1, 1)
    return (img_tensor * std + mean).clamp(0, 1)

def tensor_to_display_image(img_tensor):
    img = denormalize_tensor(img_tensor.detach()).cpu().permute(1, 2, 0).numpy()
    return img

def attribution_to_map(attr_tensor):
    # Convert any [1, C, H, W] attribution tensor into a single 2D map
    attr = attr_tensor.detach().cpu().numpy()[0]
    if attr.ndim == 3:
        attr = np.abs(attr).mean(axis=0)
    attr = attr - attr.min()
    attr = attr / (attr.max() + 1e-8)
    return attr

def get_image_tensor_from_index(global_index):
    pil_img = Image.fromarray(images[int(global_index)])
    x = eval_tfms(pil_img).unsqueeze(0).to(DEVICE)
    y = torch.tensor([int(labels[int(global_index)])], dtype=torch.long, device=DEVICE)
    return x, y

def explain_resnet_sample(global_index, target_class=None): # Runs four explanation methods on one image and shows heatmaps side by side
    x, y = get_image_tensor_from_index(global_index)
    resnet_model.eval()

    with torch.no_grad():
        logits = resnet_model(x)
        probs = F.softmax(logits, dim=1)
        pred_class = int(probs.argmax(dim=1).item())
        pred_conf = float(probs.max(dim=1).values.item())

    target = pred_class if target_class is None else int(target_class)

    saliency = Saliency(resnet_model)
    sal_attr = saliency.attribute(x, target=target)

    ig = IntegratedGradients(resnet_model)
    ig_attr = ig.attribute(x, target=target, n_steps=32)

    gradcam = LayerGradCam(resnet_model, resnet_model.backbone.layer4[-1].conv2) # Points Grad-CAM at the last conv layer where the model makes its final spatial decisions
    gc_attr = gradcam.attribute(x, target=target)
    gc_attr = LayerGradCam.interpolate(gc_attr, x.shape[-2:])

    occlusion = Occlusion(resnet_model)
    occ_attr = occlusion.attribute(
        x,
        target=target,
        sliding_window_shapes=(3, 32, 32),
        strides=(3, 16, 16),
    )

    base_img = tensor_to_display_image(x[0])

    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    axes[0].imshow(base_img)
    axes[0].set_title(
        f"Original\nTrue: {class_names[int(y.item())]}\nPred: {class_names[pred_class]} ({pred_conf:.3f})"
    )
    axes[0].axis("off")

    for ax, title, attr in [
        (axes[1], "Saliency", sal_attr),
        (axes[2], "Integrated Gradients", ig_attr),
        (axes[3], "Grad-CAM", gc_attr),
        (axes[4], "Occlusion", occ_attr),
    ]:
        heatmap = attribution_to_map(attr)
        ax.imshow(base_img)
        ax.imshow(heatmap, alpha=0.5, cmap="jet")
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    return {
        "true_class": int(y.item()),
        "pred_class": pred_class,
        "pred_confidence": pred_conf,
    }

resnet_test_pred_df = pd.DataFrame({
    "global_index": resnet_test_indices,
    "true_label": resnet_test_labels,
    "pred_label": resnet_test_probs_cal.argmax(axis=1),
    "confidence": resnet_test_probs_cal.max(axis=1),
})
resnet_test_pred_df["correct"] = (resnet_test_pred_df["true_label"] == resnet_test_pred_df["pred_label"]).astype(int)

correct_examples = resnet_test_pred_df.query("correct == 1").sort_values("confidence", ascending=False).head(2)
wrong_examples = resnet_test_pred_df.query("correct == 0").sort_values("confidence", ascending=False).head(2)

chosen_indices = correct_examples["global_index"].tolist() + wrong_examples["global_index"].tolist()
print("Explaining these global indices:", chosen_indices)

for idx in chosen_indices:
    explain_resnet_sample(int(idx))

## Section 22 — transformer interpretability (ViT)

In [ ]:
# ============================================================
# 22) TRANSFORMER INTERPRETABILITY FOR THE ViT MODEL
# ============================================================

# Disable fused attention so attn_drop hooks work
for block in vit_model.backbone.blocks:
    block.attn.fused_attn = False

def get_vit_attention_maps(model, x):
    attention_maps = []
    hooks = []

    def make_hook():
        def hook(module, inputs, output):
            if isinstance(output, torch.Tensor) and output.dim() == 4:
                attention_maps.append(output.detach())
        return hook

    for block in model.backbone.blocks:
        hooks.append(block.attn.attn_drop.register_forward_hook(make_hook()))

    model.eval()
    with torch.no_grad():
        _ = model(x)

    for h in hooks:
        h.remove()

    return attention_maps

def attention_rollout(model, x, discard_ratio=0.0):
    attention_maps = get_vit_attention_maps(model, x)

    if not attention_maps:
        raise RuntimeError("No attention maps were captured. Check the ViT block names in timm.")

    result = torch.eye(attention_maps[0].size(-1), device=x.device).unsqueeze(0)

    for attn in attention_maps:
        # attn shape: [B, heads, tokens, tokens]
        attn_fused = attn.mean(dim=1)

        if discard_ratio > 0:
            flat = attn_fused.view(attn_fused.size(0), -1)
            num_discard = int(flat.size(1) * discard_ratio)
            if num_discard > 0:
                _, indices = flat.topk(num_discard, dim=1, largest=False)
                flat.scatter_(1, indices, 0.0)
                attn_fused = flat.view_as(attn_fused)

        identity = torch.eye(attn_fused.size(-1), device=x.device).unsqueeze(0)
        attn_fused = attn_fused + identity
        attn_fused = attn_fused / attn_fused.sum(dim=-1, keepdim=True)
        result = torch.bmm(attn_fused, result)

    # CLS token attention to the image patches
    mask = result[:, 0, 1:]
    num_patches = mask.size(-1)
    grid_size = int(math.sqrt(num_patches))
    mask = mask.reshape(mask.size(0), 1, grid_size, grid_size)
    mask = F.interpolate(mask, size=(CONFIG["image_size"], CONFIG["image_size"]), mode="bilinear", align_corners=False)
    mask = mask[0, 0].detach().cpu().numpy()
    mask = mask - mask.min()
    mask = mask / (mask.max() + 1e-8)
    return mask

def explain_vit_sample(global_index, target_class=None):
    x, y = get_image_tensor_from_index(global_index)
    vit_model.eval()

    with torch.no_grad():
        logits = vit_model(x)
        probs = F.softmax(logits, dim=1)
        pred_class = int(probs.argmax(dim=1).item())
        pred_conf = float(probs.max(dim=1).values.item())

    target = pred_class if target_class is None else int(target_class)

    ig = IntegratedGradients(vit_model)
    ig_attr = ig.attribute(x, target=target, n_steps=32)
    ig_map = attribution_to_map(ig_attr)

    attn_map = attention_rollout(vit_model, x, discard_ratio=0.0)

    base_img = tensor_to_display_image(x[0])

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(base_img)
    axes[0].set_title(
        f"Original\nTrue: {class_names[int(y.item())]}\nPred: {class_names[pred_class]} ({pred_conf:.3f})"
    )
    axes[0].axis("off")

    axes[1].imshow(base_img)
    axes[1].imshow(ig_map, alpha=0.5, cmap="jet")
    axes[1].set_title("ViT Integrated Gradients")
    axes[1].axis("off")

    axes[2].imshow(base_img)
    axes[2].imshow(attn_map, alpha=0.5, cmap="jet")
    axes[2].set_title("ViT Attention Rollout")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

vit_test_pred_df = pd.DataFrame({
    "global_index": vit_test_indices,
    "true_label": vit_test_labels,
    "pred_label": vit_test_probs_cal.argmax(axis=1),
    "confidence": vit_test_probs_cal.max(axis=1),
})
vit_test_pred_df["correct"] = (vit_test_pred_df["true_label"] == vit_test_pred_df["pred_label"]).astype(int)

correct_examples_vit = vit_test_pred_df.query("correct == 1").sort_values("confidence", ascending=False).head(2)
wrong_examples_vit = vit_test_pred_df.query("correct == 0").sort_values("confidence", ascending=False).head(2)

chosen_indices_vit = correct_examples_vit["global_index"].tolist() + wrong_examples_vit["global_index"].tolist()
print("Explaining these global indices with ViT:", chosen_indices_vit)

for idx in chosen_indices_vit:
    explain_vit_sample(int(idx))

## Section 23 — tabular interpretability

In [ ]:
# ============================================================
# 23) TABULAR INTERPRETABILITY
# ============================================================
# We use:
# - permutation importance,
# - SHAP,
# - partial dependence / ICE on the top features.

rf_interpret_model = RandomForestClassifier( # Trains a separate random forest just for explainability (SHAP, permutation importance, PDP)
    n_estimators=400 if not FAST_RUN else 150,
    random_state=SEED,
    n_jobs=-1,
    class_weight="balanced_subsample",
)
rf_interpret_model.fit(X_tab_train, y_train)

# Permutation importance
perm = permutation_importance(
    rf_interpret_model,
    X_tab_test,
    y_test,
    n_repeats=10 if FAST_RUN else 20,
    random_state=SEED,
    scoring="f1_macro",
    n_jobs=-1,
)

perm_df = pd.DataFrame({
    "feature": feature_columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

display(perm_df.head(20))

plt.figure(figsize=(10, 6))
top_perm = perm_df.head(15).iloc[::-1]
plt.barh(top_perm["feature"], top_perm["importance_mean"])
plt.title("Permutation importance for the tabular model")
plt.xlabel("Mean drop in macro F1 after permutation")
plt.tight_layout()
plt.show()

# SHAP on a manageable subset
shap_sample_size = min(200, len(X_tab_test))
shap_idx = np.random.choice(np.arange(len(X_tab_test)), size=shap_sample_size, replace=False)
X_shap = X_tab_test[shap_idx]

explainer = shap.TreeExplainer(rf_interpret_model)
shap_values = explainer.shap_values(X_shap)

# Handle different SHAP output formats for multiclass models
if isinstance(shap_values, list):
    mean_abs_shap = np.mean(np.stack([np.abs(sv) for sv in shap_values], axis=0), axis=0)
else:
    shap_arr = np.array(shap_values)
    if shap_arr.ndim == 3:
        mean_abs_shap = np.mean(np.abs(shap_arr), axis=2)
    else:
        mean_abs_shap = np.abs(shap_arr)

shap_importance = pd.DataFrame({
    "feature": feature_columns,
    "mean_abs_shap": mean_abs_shap.mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

display(shap_importance.head(20))

plt.figure(figsize=(10, 6))
top_shap = shap_importance.head(15).iloc[::-1]
plt.barh(top_shap["feature"], top_shap["mean_abs_shap"])
plt.title("SHAP feature importance for the tabular model")
plt.xlabel("Mean absolute SHAP value")
plt.tight_layout()
plt.show()

# PDP / ICE for a few top features
top_feature_names = shap_importance["feature"].head(3).tolist()
target_class_for_pdp = 0
for i, name in enumerate(class_names):
    if "tumor" in name.lower() or "tumour" in name.lower():
        target_class_for_pdp = i
        break

X_tab_train_df = pd.DataFrame(X_tab_train, columns=feature_columns)

for feature_name in top_feature_names:
    fig, ax = plt.subplots(figsize=(6, 4))
    PartialDependenceDisplay.from_estimator(
        rf_interpret_model,
        X_tab_train_df,
        [feature_name],
        target=target_class_for_pdp,
        kind="both",
        ax=ax,
    )
    ax.set_title(f"PDP + ICE for {feature_name}\n(target class: {class_names[target_class_for_pdp]})")
    plt.tight_layout()
    plt.show()

## Section 24 — fusion interpretability

In [ ]:
# ============================================================
# 24) FUSION INTERPRETABILITY
# ============================================================
# We inspect:
# - gate weights in the gated fusion model,
# - modality ablation,
# - cross-attention weights in the attention fusion model.

gated_model = trained_fusion_models["GatedFusion"].to(DEVICE)
crossattn_model = trained_fusion_models["CrossAttentionFusion"].to(DEVICE)

def collect_gates(model, loader):
    model.eval()
    gate_rows = []
    with torch.no_grad():
        for batch in loader:
            img_feat = batch["img_emb"].to(DEVICE)
            tab_feat = batch["tab"].to(DEVICE)
            y = batch["label"].cpu().numpy()

            logits, parts = model(img_feat, tab_feat, return_parts=True)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            gates = parts["gates"].cpu().numpy()

            for true_label, prob_vec, gate_vec in zip(y, probs, gates):
                gate_rows.append({
                    "true_label": int(true_label),
                    "pred_label": int(prob_vec.argmax()),
                    "confidence": float(prob_vec.max()),
                    "image_gate": float(gate_vec[0]),
                    "tabular_gate": float(gate_vec[1]),
                })
    return pd.DataFrame(gate_rows)

gate_df = collect_gates(gated_model, test_loader_fusion)
gate_df["true_class_name"] = gate_df["true_label"].map({i: n for i, n in enumerate(class_names)})
gate_df["correct"] = (gate_df["true_label"] == gate_df["pred_label"]).astype(int)

display(gate_df.head())

gate_summary = gate_df.groupby("true_class_name")[["image_gate", "tabular_gate"]].mean().sort_values("image_gate", ascending=False)
display(gate_summary)

plt.figure(figsize=(10, 5))
x = np.arange(len(gate_summary))
plt.bar(x - 0.2, gate_summary["image_gate"].values, width=0.4, label="image")
plt.bar(x + 0.2, gate_summary["tabular_gate"].values, width=0.4, label="tabular")
plt.xticks(x, gate_summary.index, rotation=45, ha="right")
plt.ylabel("Mean gate weight")
plt.title("Average gate weight by true class")
plt.legend()
plt.tight_layout()
plt.show()

def evaluate_with_modality_ablation(model, loader, zero_image=False, zero_tab=False):
    model.eval()
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            img_feat = batch["img_emb"].to(DEVICE)
            tab_feat = batch["tab"].to(DEVICE)
            y = batch["label"].cpu().numpy()

            if zero_image:
                img_feat = torch.zeros_like(img_feat)
            if zero_tab:
                tab_feat = torch.zeros_like(tab_feat)

            logits = model(img_feat, tab_feat)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(y)

    probs = np.vstack(all_probs)
    labels_np = np.concatenate(all_labels)
    return compute_metrics(labels_np, probs, model_name="ablation")

full_metrics = evaluate_with_modality_ablation(gated_model, test_loader_fusion, zero_image=False, zero_tab=False) # Baseline: both modalities present
image_only_metrics = evaluate_with_modality_ablation(gated_model, test_loader_fusion, zero_image=False, zero_tab=True) # Zeros out tabular to see how much image alone contributes
tab_only_metrics = evaluate_with_modality_ablation(gated_model, test_loader_fusion, zero_image=True, zero_tab=False) # Zeros out image to see how much tabular alone contributes

ablation_df = pd.DataFrame([
    {"setting": "full multimodal", **full_metrics}, # Both modalities used
    {"setting": "image kept, tabular removed", **image_only_metrics}, # Only image signal
    {"setting": "tabular kept, image removed", **tab_only_metrics}, # Only tabular signal
])
display(ablation_df)

# Cross-attention weights
def collect_cross_attention(model, loader):
    model.eval()
    attn_list = []
    with torch.no_grad():
        for batch in loader:
            img_feat = batch["img_emb"].to(DEVICE)
            tab_feat = batch["tab"].to(DEVICE)
            _, parts = model(img_feat, tab_feat, return_parts=True)
            attn = parts["attention_weights"].detach().cpu().numpy()  # [B, heads, 2, 2]
            attn_list.append(attn)
    return np.concatenate(attn_list, axis=0)

cross_attn_weights = collect_cross_attention(crossattn_model, test_loader_fusion)
mean_cross_attn = cross_attn_weights.mean(axis=(0, 1))

plt.figure(figsize=(4, 4))
plt.imshow(mean_cross_attn, cmap="viridis")
plt.xticks([0, 1], ["image_token", "tabular_token"])
plt.yticks([0, 1], ["image_token", "tabular_token"])
plt.title("Mean cross-attention matrix")
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{mean_cross_attn[i, j]:.3f}", ha="center", va="center")
plt.colorbar()
plt.tight_layout()
plt.show()

## Section 25 — optional LIME explanations

In [ ]:
# ============================================================
# 25) OPTIONAL MODEL-AGNOSTIC LIME EXPLANATIONS
# ============================================================
# This section is slower, so it is disabled in FAST_RUN by default.

if CONFIG["run_lime"]:
    from lime import lime_image, lime_tabular
    from skimage.segmentation import mark_boundaries

    # -------------------------
    # Image LIME
    # -------------------------
    def resnet_predict_for_lime(image_list):
        batch_tensors = []
        for img in image_list:
            img_uint8 = np.uint8(np.clip(img, 0, 255))
            pil_img = Image.fromarray(img_uint8)
            batch_tensors.append(eval_tfms(pil_img))
        batch = torch.stack(batch_tensors).to(DEVICE)
        with torch.no_grad():
            probs = F.softmax(resnet_model(batch), dim=1).cpu().numpy()
        return probs

    lime_img_explainer = lime_image.LimeImageExplainer()
    lime_global_idx = int(resnet_test_pred_df.sort_values("confidence", ascending=False).iloc[0]["global_index"])
    lime_image_np = images[lime_global_idx]

    lime_exp_img = lime_img_explainer.explain_instance(
        lime_image_np.astype(np.double),
        classifier_fn=resnet_predict_for_lime,
        top_labels=1,
        hide_color=0,
        num_samples=500,
    )

    temp_img, temp_mask = lime_exp_img.get_image_and_mask(
        label=lime_exp_img.top_labels[0],
        positive_only=True,
        num_features=10,
        hide_rest=False,
    )

    plt.figure(figsize=(6, 6))
    plt.imshow(mark_boundaries(temp_img / 255.0 if temp_img.max() > 1 else temp_img, temp_mask))
    plt.title("LIME image explanation (ResNet18)")
    plt.axis("off")
    plt.show()

    # -------------------------
    # Tabular LIME
    # -------------------------
    lime_tab_explainer = lime_tabular.LimeTabularExplainer(
        training_data=X_tab_train,
        feature_names=feature_columns,
        class_names=class_names,
        mode="classification",
        random_state=SEED,
    )

    sample_id = 0
    lime_exp_tab = lime_tab_explainer.explain_instance(
        X_tab_test[sample_id],
        rf_model.predict_proba,
        num_features=10,
        top_labels=1,
    )

    print("LIME tabular explanation for one test case:")
    for item in lime_exp_tab.as_list(label=lime_exp_tab.top_labels[0]):
        print(item)
else:
    print("LIME section skipped. Set CONFIG['run_lime'] = True to enable it.")

## Section 26 — error analysis and test case review

In [ ]:
# ============================================================
# 26) ERROR ANALYSIS AND TEST CASE REVIEW
# ============================================================

prediction_bank = {
    "rf": rf_test_probs,
    "hog": hog_test_probs,
    "resnet": resnet_test_probs_cal,
    "vit": vit_test_probs_cal,
    "late_fusion": late_fusion_test_probs,
    "stacking": stack_test_probs,
}

for fusion_name, probs in fusion_test_probs_cal.items():
    prediction_bank[fusion_name] = probs

fusion_result_names = [name for name in results_df["model"].tolist() if "Fusion" in name or "ensemble" in name or "LateFusion" in name]
print("Fusion or ensemble model names found:", fusion_result_names)

result_name_to_bank_key = {
    "Tabular RandomForest (calibrated)": "rf",
    "Image HOG + LinearSVM (calibrated)": "hog",
    "ResNet18 raw": "resnet",
    "ResNet18 calibrated": "resnet",
    "ViT-tiny raw": "vit",
    "ViT-tiny calibrated": "vit",
    "LateFusion weighted average": "late_fusion",
    "Stacking ensemble": "stacking",
}
for fusion_name in fusion_test_probs_cal:
    result_name_to_bank_key[f"{fusion_name} calibrated"] = fusion_name

 # Add student-added models
prediction_bank["logreg"] = logreg_test_probs
prediction_bank["mobilenet"] = extra_test_probs_cal

result_name_to_bank_key["Tabular LogisticRegression"] = "logreg"
result_name_to_bank_key["MobileNetV3-Small raw"] = "mobilenet"
result_name_to_bank_key["MobileNetV3-Small calibrated"] = "mobilenet"

best_model_key = result_name_to_bank_key.get(best_model_name, "stacking")
best_test_probs = prediction_bank[best_model_key]
best_test_preds = best_test_probs.argmax(axis=1)
best_test_conf = best_test_probs.max(axis=1)

error_df = pd.DataFrame({
    "global_index": test_idx,
    "true_label": y_test,
    "true_class": [class_names[y] for y in y_test],
    "pred_label": best_test_preds,
    "pred_class": [class_names[p] for p in best_test_preds],
    "confidence": best_test_conf,
})
error_df["correct"] = (error_df["true_label"] == error_df["pred_label"]).astype(int)

for bank_key, probs in prediction_bank.items():
    preds = probs.argmax(axis=1)
    confs = probs.max(axis=1)
    error_df[f"{bank_key}_pred"] = [class_names[p] for p in preds]
    error_df[f"{bank_key}_conf"] = confs

display(error_df.head(20))

per_class_rows = []
for class_id, class_name in enumerate(class_names):
    class_mask = error_df["true_label"].values == class_id
    class_acc = (error_df.loc[class_mask, "pred_label"].values == class_id).mean()
    per_class_rows.append({
        "class_id": class_id,
        "class_name": class_name,
        "test_count": int(class_mask.sum()),
        "class_accuracy": float(class_acc),
    })

per_class_df = pd.DataFrame(per_class_rows).sort_values("class_accuracy")
display(per_class_df)

plt.figure(figsize=(10, 4))
plt.bar(per_class_df["class_name"], per_class_df["class_accuracy"])
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.title(f"Per-class accuracy for the best model: {best_model_name}")
plt.tight_layout()
plt.show()

cm_best = confusion_matrix(y_test, best_test_preds, labels=list(range(num_classes)))
confusion_rows = []
for i in range(num_classes):
    for j in range(num_classes):
        if i != j and cm_best[i, j] > 0:
            confusion_rows.append({
                "true_class": class_names[i],
                "pred_class": class_names[j],
                "count": int(cm_best[i, j]),
            })

confusion_pairs_df = pd.DataFrame(confusion_rows).sort_values("count", ascending=False)
display(confusion_pairs_df.head(15))

def show_test_case(global_index, title_prefix=""):
    img = images[int(global_index)]
    hematoxylin, smooth, mask, labels_ws = segment_nuclei_from_histology(img)
    boxes = labels_to_boxes(labels_ws)

    row = error_df.loc[error_df["global_index"] == global_index].iloc[0]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(img)
    axes[0].set_title(
        f"{title_prefix}Original\nTrue: {row['true_class']}\nBest pred: {row['pred_class']} ({row['confidence']:.3f})"
    )
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("Classical segmentation mask")
    axes[1].axis("off")

    axes[2].imshow(img)
    for box in boxes[:30]:
        rect = plt.Rectangle(
            (box["x1"], box["y1"]),
            box["x2"] - box["x1"],
            box["y2"] - box["y1"],
            fill=False,
            linewidth=1.0,
        )
        axes[2].add_patch(rect)
    axes[2].set_title(f"Detection overlay ({len(boxes)} boxes)")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

    compare_cols = [
        "rf_pred", "hog_pred", "resnet_pred", "vit_pred", "late_fusion_pred", "stacking_pred"
    ]
    compare_conf_cols = [
        "rf_conf", "hog_conf", "resnet_conf", "vit_conf", "late_fusion_conf", "stacking_conf"
    ]
    comparison = pd.DataFrame({
        "model": compare_cols,
        "prediction": [row[c] for c in compare_cols],
        "confidence": [row[c] for c in compare_conf_cols],
    })
    display(comparison)

confident_errors = error_df.query("correct == 0").sort_values("confidence", ascending=False).head(4)
uncertain_correct = error_df.query("correct == 1").sort_values("confidence", ascending=True).head(4)

print("Confident errors")
if len(confident_errors) > 0:
    display(confident_errors[["global_index", "true_class", "pred_class", "confidence"]])
    for idx in confident_errors["global_index"].tolist():
        show_test_case(int(idx), title_prefix="Confident error | ")
else:
    print("No confident errors were found in this run.")

print("Uncertain but correct cases")
if len(uncertain_correct) > 0:
    display(uncertain_correct[["global_index", "true_class", "pred_class", "confidence"]])
    for idx in uncertain_correct["global_index"].tolist():
        show_test_case(int(idx), title_prefix="Uncertain correct | ")
else:
    print("No uncertain correct cases were found in this run.")

## Section 27 — detailed multimodal case study

In [ ]:
# ============================================================
# 27) DETAILED MULTIMODAL CASE STUDY
# ============================================================
# We combine:
# - the image,
# - segmentation and detections,
# - the CNN/ViT explanations,
# - local tabular SHAP,
# - gate weights from the fusion model.

def get_local_shap_for_sample(sample_global_index):
    sample_pos = np.where(test_idx == sample_global_index)[0]
    if len(sample_pos) == 0:
        raise ValueError("Sample is not in the test split.")
    sample_pos = int(sample_pos[0])
    x_sample = X_tab_test[sample_pos:sample_pos + 1]

    local_shap_values = explainer.shap_values(x_sample)
    if isinstance(local_shap_values, list):
        # Use the predicted class from the RF interpretability model
        pred_class = int(rf_interpret_model.predict(x_sample)[0])
        local_vals = local_shap_values[pred_class][0]
    else:
        local_arr = np.array(local_shap_values)
        if local_arr.ndim == 3:
            pred_class = int(rf_interpret_model.predict(x_sample)[0])
            local_vals = local_arr[0, :, pred_class]
        else:
            local_vals = local_arr[0]
    local_df = pd.DataFrame({
        "feature": feature_columns,
        "shap_value": local_vals,
        "feature_value": x_sample[0],
    }).sort_values("shap_value", key=np.abs, ascending=False)
    return local_df

def get_gate_for_sample(sample_global_index):
    sample_pos = np.where(test_idx == sample_global_index)[0]
    sample_pos = int(sample_pos[0])

    img_feat = torch.tensor(vit_emb_test[sample_pos:sample_pos + 1], dtype=torch.float32, device=DEVICE)
    tab_feat = torch.tensor(X_tab_test[sample_pos:sample_pos + 1], dtype=torch.float32, device=DEVICE)

    gated_model.eval()
    with torch.no_grad():
        logits, parts = gated_model(img_feat, tab_feat, return_parts=True)
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
        gates = parts["gates"].cpu().numpy()[0]

    return probs, gates

case_idx = int(confident_errors.iloc[0]["global_index"]) if len(confident_errors) > 0 else int(test_idx[0])

print("Case study global index:", case_idx)
show_test_case(case_idx, title_prefix="Case study | ")

local_shap_df = get_local_shap_for_sample(case_idx)
display(local_shap_df.head(12))

plt.figure(figsize=(8, 5))
top_local = local_shap_df.head(10).iloc[::-1]
plt.barh(top_local["feature"], top_local["shap_value"])
plt.title("Local tabular SHAP explanation")
plt.xlabel("Contribution toward the chosen class score")
plt.tight_layout()
plt.show()

fusion_probs, fusion_gates = get_gate_for_sample(case_idx)
print("Gated fusion probabilities:")
for i, p in enumerate(fusion_probs):
    print(f"  {class_names[i]}: {p:.4f}")
print(f"Image gate:   {fusion_gates[0]:.4f}")
print(f"Tabular gate: {fusion_gates[1]:.4f}")

print("ResNet explanation")
_ = explain_resnet_sample(case_idx)

print("ViT explanation")
_ = explain_vit_sample(case_idx)

## Section 28 — optional grading or regression version

In [ ]:
# ============================================================
# 28) OPTIONAL: HOW TO TURN THIS INTO GRADING OR REGRESSION
# ============================================================
# This notebook runs as classification out of the box.
# If you later switch to a cancer grading dataset, use one of these heads.

class OrdinalHead(nn.Module):
    """
    CORAL-style ordinal head.
    Example:
        grades 0,1,2,3 become three ordered binary decisions:
        grade > 0, grade > 1, grade > 2
    """
    def __init__(self, in_features, num_grades):
        super().__init__()
        self.fc = nn.Linear(in_features, 1, bias=False)
        self.bias = nn.Parameter(torch.zeros(num_grades - 1))
        self.num_grades = num_grades

    def forward(self, x):
        logits = self.fc(x) + self.bias
        return logits  # shape [B, num_grades - 1]

def grades_to_levels(y, num_grades):
    """
    Convert grade labels like [0, 2, 3] into ordered binary vectors.
    For num_grades=4:
      0 -> [0,0,0]
      1 -> [1,0,0]
      2 -> [1,1,0]
      3 -> [1,1,1]
    """
    y = torch.as_tensor(y, dtype=torch.long)
    levels = []
    for grade in y:
        levels.append([1 if grade > k else 0 for k in range(num_grades - 1)])
    return torch.tensor(levels, dtype=torch.float32, device=y.device)

def coral_loss(logits, levels):
    return F.binary_cross_entropy_with_logits(logits, levels)

def ordinal_logits_to_grade(logits):
    probs = torch.sigmoid(logits)
    return (probs > 0.5).sum(dim=1)

# Regression version
class RegressionHead(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.fc = nn.Linear(in_features, 1)

    def forward(self, x):
        return self.fc(x).squeeze(1)

print(
    "This section is a template. "
    "To use it, replace the classification head in any image or fusion model, "
    "then swap CrossEntropyLoss for either coral_loss (grading) or SmoothL1Loss (regression)."
)

## Section 23B — SHAP beeswarm plot

In [ ]:
# ============================================================
# SHAP Beeswarm Plot
# ============================================================

# Normalize shap_values into a 2D array (samples x features)
# averaged across all classes — matches how your bar chart works
if isinstance(shap_values, list):
    # List of [samples x features] arrays, one per class
    shap_2d = np.mean(np.stack(shap_values, axis=0), axis=0)        # shape: (samples, features)
else:
    shap_arr = np.array(shap_values)
    if shap_arr.ndim == 3:                                           # shape: (samples, features, classes)
        shap_2d = np.mean(shap_arr, axis=2)
    else:
        shap_2d = shap_arr

# Build an Explanation object so shap.plots.beeswarm works correctly
explanation = shap.Explanation(
    values=shap_2d,
    data=X_shap,
    feature_names=feature_columns,
)

# Beeswarm — top 15 features
plt.figure(figsize=(10, 7))
shap.plots.beeswarm(
    explanation,
    max_display=15,
    show=False,
)
plt.title("SHAP Beeswarm — tabular model (mean over classes)", fontsize=13)
plt.tight_layout()
plt.show()

## Section 29 — save all figures and download as ZIP

In [ ]:
# ============================================================
# 29) SAVE ALL FIGURES TO PNG AND DOWNLOAD AS ZIP
# ============================================================
# This cell regenerates every plot from the notebook using
# variables already in memory, saves each as a high-resolution
# PNG, and packages them into a single ZIP for download.
#
# Run this AFTER all previous cells have completed successfully.

import os, zipfile, shutil
from google.colab import files as colab_files

FIGURE_DIR = "assignment1_figures"
if os.path.exists(FIGURE_DIR):
    shutil.rmtree(FIGURE_DIR)
os.makedirs(FIGURE_DIR, exist_ok=True)

DPI = 200
fig_counter = [0]

def save_fig(name, fig=None, dpi=DPI):
    """Save the current (or given) figure and close it."""
    fig_counter[0] += 1
    prefix = f"{fig_counter[0]:02d}"
    path = os.path.join(FIGURE_DIR, f"{prefix}_{name}.png")
    if fig is not None:
        fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
        plt.close(fig)
    else:
        plt.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
        plt.close()
    print(f"  Saved: {path}")

print("=" * 60)
print("SAVING ALL FIGURES")
print("=" * 60)

# ------------------------------------------------------------------
# Section 4 - Class distribution
# ------------------------------------------------------------------
print("\n[Section 4] Basic exploration")

plt.figure(figsize=(12, 4))
plt.bar(label_df["class_name"], label_df["count"])
plt.xticks(rotation=45, ha="right")
plt.title("Class distribution")
plt.ylabel("Number of images")
plt.tight_layout()
save_fig("class_distribution")

# Sample images grid
np.random.seed(SEED)
n_per_class = 4
n_classes = len(class_names)
fig, axes = plt.subplots(n_classes, n_per_class, figsize=(3 * n_per_class, 3 * n_classes))
if n_classes == 1:
    axes = np.array([axes])
for class_id in range(n_classes):
    class_indices = np.where(labels == class_id)[0]
    chosen = np.random.choice(class_indices, size=min(n_per_class, len(class_indices)), replace=False)
    for j in range(n_per_class):
        ax = axes[class_id, j] if n_classes > 1 else axes[j]
        ax.axis("off")
        if j < len(chosen):
            ax.imshow(images[chosen[j]])
            if j == 0:
                ax.set_title(f"{class_names[class_id]}")
plt.suptitle("Sample images from each class", y=1.02, fontsize=14)
plt.tight_layout()
save_fig("sample_images_per_class")

# ------------------------------------------------------------------
# Section 5 - Segmentation examples
# ------------------------------------------------------------------
print("\n[Section 5] Segmentation and detection examples")
np.random.seed(SEED)
seg_sample_ids = np.random.choice(np.arange(len(images)), size=6, replace=False)
for k, idx in enumerate(seg_sample_ids):
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    img_rgb = images[idx]
    hematoxylin, smooth, mask, labels_ws = segment_nuclei_from_histology(img_rgb)
    boxes = labels_to_boxes(labels_ws)
    overlay = img_rgb.copy()
    axes[0].imshow(img_rgb); axes[0].set_title("Original"); axes[0].axis("off")
    axes[1].imshow(hematoxylin, cmap="gray"); axes[1].set_title("Hematoxylin"); axes[1].axis("off")
    axes[2].imshow(mask, cmap="gray"); axes[2].set_title("Segmentation mask"); axes[2].axis("off")
    axes[3].imshow(overlay); axes[3].set_title(f"Detection ({len(boxes)} boxes)"); axes[3].axis("off")
    for box in boxes[:30]:
        rect = plt.Rectangle((box["x1"], box["y1"]), box["x2"]-box["x1"], box["y2"]-box["y1"],
                              fill=False, linewidth=1.0)
        axes[3].add_patch(rect)
    plt.suptitle(f"Example {idx} | label = {class_names[labels[idx]]}")
    plt.tight_layout()
    save_fig(f"segmentation_example_{k+1}")

# ------------------------------------------------------------------
# Section 7 - Tabular feature boxplots
# ------------------------------------------------------------------
print("\n[Section 7] Tabular feature inspection")
plot_features = [c for c in ["nuclei_count", "nuclei_pixel_ratio", "nucleus_area_mean",
                              "nucleus_eccentricity_mean", "glcm_contrast"]
                 if c in feature_df.columns]
if plot_features:
    fig, axes = plt.subplots(1, len(plot_features), figsize=(5 * len(plot_features), 4))
    if len(plot_features) == 1:
        axes = [axes]
    for ax, col_name in zip(axes, plot_features):
        grouped = []
        labels_local = []
        for class_id, class_name in enumerate(class_names):
            vals = feature_df.loc[feature_df["label"] == class_id, col_name].values
            grouped.append(vals)
            labels_local.append(class_name)
        ax.boxplot(grouped, labels=labels_local, showfliers=False)
        ax.set_title(col_name)
        ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    save_fig("tabular_feature_boxplots")

# ------------------------------------------------------------------
# Section 10 - Classical baselines
# ------------------------------------------------------------------
print("\n[Section 10] Classical AI baselines")

# RandomForest
preds_rf = rf_test_probs.argmax(axis=1)
cm_rf = confusion_matrix(y_test, preds_rf)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=class_names).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("Tabular RandomForest confusion matrix")
plt.tight_layout()
save_fig("rf_confusion_matrix")

plot_reliability_diagram(rf_test_probs, y_test, title="Tabular RandomForest reliability")
save_fig("rf_reliability")

# HOG + SVM
preds_hog = hog_test_probs.argmax(axis=1)
cm_hog = confusion_matrix(y_test, preds_hog)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(confusion_matrix=cm_hog, display_labels=class_names).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("HOG + LinearSVM confusion matrix")
plt.tight_layout()
save_fig("hog_svm_confusion_matrix")

plot_reliability_diagram(hog_test_probs, y_test, title="HOG + LinearSVM reliability")
save_fig("hog_svm_reliability")

# ------------------------------------------------------------------
# Section 10C - Logistic Regression
# ------------------------------------------------------------------
print("\n[Section 10C] Logistic Regression")

preds_lr = logreg_test_probs.argmax(axis=1)
cm_lr = confusion_matrix(y_test, preds_lr)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=class_names).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("Tabular LogisticRegression confusion matrix")
plt.tight_layout()
save_fig("logreg_confusion_matrix")

plot_reliability_diagram(logreg_test_probs, y_test, title="Tabular LogisticRegression reliability")
save_fig("logreg_reliability")

# ------------------------------------------------------------------
# Section 13 - ResNet18
# ------------------------------------------------------------------
print("\n[Section 13] ResNet18")

plot_history(resnet_history, title="ResNet18")
save_fig("resnet18_training_history")

preds_resnet = resnet_test_probs_cal.argmax(axis=1)
cm_resnet = confusion_matrix(resnet_test_labels, preds_resnet)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(confusion_matrix=cm_resnet, display_labels=class_names).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("ResNet18 calibrated confusion matrix")
plt.tight_layout()
save_fig("resnet18_confusion_matrix")

plot_reliability_diagram(resnet_test_probs_raw, resnet_test_labels, title="ResNet18 reliability before calibration")
save_fig("resnet18_reliability_before_cal")

plot_reliability_diagram(resnet_test_probs_cal, resnet_test_labels, title="ResNet18 reliability after calibration")
save_fig("resnet18_reliability_after_cal")

# ------------------------------------------------------------------
# Section 14 - ViT-tiny
# ------------------------------------------------------------------
print("\n[Section 14] ViT-tiny")

plot_history(vit_history, title="ViT-tiny")
save_fig("vit_training_history")

preds_vit = vit_test_probs_cal.argmax(axis=1)
cm_vit = confusion_matrix(vit_test_labels, preds_vit)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(confusion_matrix=cm_vit, display_labels=class_names).plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("ViT-tiny calibrated confusion matrix")
plt.tight_layout()
save_fig("vit_confusion_matrix")

plot_reliability_diagram(vit_test_probs_raw, vit_test_labels, title="ViT-tiny reliability before calibration")
save_fig("vit_reliability_before_cal")

plot_reliability_diagram(vit_test_probs_cal, vit_test_labels, title="ViT-tiny reliability after calibration")
save_fig("vit_reliability_after_cal")

# ------------------------------------------------------------------
# Section 14B - MobileNetV3-Small
# ------------------------------------------------------------------
print("\n[Section 14B] MobileNetV3-Small")

plot_history(extra_history, title="MobileNetV3-Small")
save_fig("mobilenet_training_history")

# ------------------------------------------------------------------
# Section 18 - Fusion models (training + reliability)
# ------------------------------------------------------------------
print("\n[Section 18] Fusion models")
for fusion_name in fusion_histories:
    plot_history(fusion_histories[fusion_name], title=fusion_name)
    save_fig(f"fusion_{fusion_name}_training_history")

    if fusion_name in fusion_test_probs_cal:
        test_labels_f = fusion_test_logits[fusion_name]  # need labels
        # Re-collect labels from the loader
        _all_labels = []
        for batch in test_loader_fusion:
            _all_labels.append(batch["label"].numpy())
        _test_labels_f = np.concatenate(_all_labels)
        plot_reliability_diagram(
            fusion_test_probs_cal[fusion_name], _test_labels_f,
            title=f"{fusion_name} reliability after calibration"
        )
        save_fig(f"fusion_{fusion_name}_reliability")

# ------------------------------------------------------------------
# Section 19 - Late fusion & stacking
# ------------------------------------------------------------------
print("\n[Section 19] Late fusion and stacking")

plot_reliability_diagram(late_fusion_test_probs, y_test, title="LateFusion reliability")
save_fig("late_fusion_reliability")

plot_reliability_diagram(stack_test_probs, y_test, title="Stacking reliability")
save_fig("stacking_reliability")

# ------------------------------------------------------------------
# Section 20 - Model comparison bar charts
# ------------------------------------------------------------------
print("\n[Section 20] Model comparison charts")
metric_cols = ["accuracy", "macro_f1", "balanced_accuracy", "macro_ovr_auc", "log_loss", "ece", "brier"]
for metric in metric_cols:
    plt.figure(figsize=(10, 4))
    plt.bar(results_df["model"], results_df[metric])
    plt.xticks(rotation=60, ha="right")
    plt.title(f"Model comparison: {metric}")
    plt.tight_layout()
    save_fig(f"model_comparison_{metric}")

# ------------------------------------------------------------------
# Section 21 - ResNet18 image interpretability
# ------------------------------------------------------------------
print("\n[Section 21] ResNet18 image interpretability (Saliency, IG, Grad-CAM, Occlusion)")

resnet_test_pred_df = pd.DataFrame({
    "global_index": resnet_test_indices,
    "true_label": resnet_test_labels,
    "pred_label": resnet_test_probs_cal.argmax(axis=1),
    "confidence": resnet_test_probs_cal.max(axis=1),
})
resnet_test_pred_df["correct"] = (resnet_test_pred_df["true_label"] == resnet_test_pred_df["pred_label"]).astype(int)

correct_examples = resnet_test_pred_df.query("correct == 1").sort_values("confidence", ascending=False).head(2)
wrong_examples = resnet_test_pred_df.query("correct == 0").sort_values("confidence", ascending=False).head(2)
chosen_indices = correct_examples["global_index"].tolist() + wrong_examples["global_index"].tolist()

for k, idx in enumerate(chosen_indices):
    x, y_lbl = get_image_tensor_from_index(int(idx))
    resnet_model.eval()
    with torch.no_grad():
        logits = resnet_model(x)
        probs = F.softmax(logits, dim=1)
        pred_class = int(probs.argmax(dim=1).item())
        pred_conf = float(probs.max(dim=1).values.item())
    target = pred_class

    saliency = Saliency(resnet_model)
    sal_attr = saliency.attribute(x, target=target)
    ig = IntegratedGradients(resnet_model)
    ig_attr = ig.attribute(x, target=target, n_steps=32)
    gradcam = LayerGradCam(resnet_model, resnet_model.backbone.layer4[-1].conv2)
    gc_attr = gradcam.attribute(x, target=target)
    gc_attr = LayerGradCam.interpolate(gc_attr, x.shape[-2:])
    occlusion = Occlusion(resnet_model)
    occ_attr = occlusion.attribute(x, target=target, sliding_window_shapes=(3, 32, 32), strides=(3, 16, 16))

    base_img = tensor_to_display_image(x[0])
    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    axes[0].imshow(base_img)
    axes[0].set_title(f"Original\nTrue: {class_names[int(y_lbl.item())]}\nPred: {class_names[pred_class]} ({pred_conf:.3f})")
    axes[0].axis("off")
    for ax, title, attr in [(axes[1], "Saliency", sal_attr), (axes[2], "Integrated Gradients", ig_attr),
                             (axes[3], "Grad-CAM", gc_attr), (axes[4], "Occlusion", occ_attr)]:
        heatmap = attribution_to_map(attr)
        ax.imshow(base_img); ax.imshow(heatmap, alpha=0.5, cmap="jet")
        ax.set_title(title); ax.axis("off")
    plt.tight_layout()
    label = "correct" if k < 2 else "error"
    save_fig(f"resnet_interpretability_{label}_{k+1}")

# ------------------------------------------------------------------
# Section 22 - ViT interpretability
# ------------------------------------------------------------------
print("\n[Section 22] ViT interpretability (IG + Attention Rollout)")

vit_test_pred_df = pd.DataFrame({
    "global_index": vit_test_indices,
    "true_label": vit_test_labels,
    "pred_label": vit_test_probs_cal.argmax(axis=1),
    "confidence": vit_test_probs_cal.max(axis=1),
})
vit_test_pred_df["correct"] = (vit_test_pred_df["true_label"] == vit_test_pred_df["pred_label"]).astype(int)
correct_examples_vit = vit_test_pred_df.query("correct == 1").sort_values("confidence", ascending=False).head(2)
wrong_examples_vit = vit_test_pred_df.query("correct == 0").sort_values("confidence", ascending=False).head(2)
chosen_indices_vit = correct_examples_vit["global_index"].tolist() + wrong_examples_vit["global_index"].tolist()

for k, idx in enumerate(chosen_indices_vit):
    x, y_lbl = get_image_tensor_from_index(int(idx))
    vit_model.eval()
    with torch.no_grad():
        logits = vit_model(x)
        probs = F.softmax(logits, dim=1)
        pred_class = int(probs.argmax(dim=1).item())
        pred_conf = float(probs.max(dim=1).values.item())
    target = pred_class

    ig = IntegratedGradients(vit_model)
    ig_attr = ig.attribute(x, target=target, n_steps=32)
    ig_map = attribution_to_map(ig_attr)
    attn_map = attention_rollout(vit_model, x, discard_ratio=0.0)
    base_img = tensor_to_display_image(x[0])

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(base_img)
    axes[0].set_title(f"Original\nTrue: {class_names[int(y_lbl.item())]}\nPred: {class_names[pred_class]} ({pred_conf:.3f})")
    axes[0].axis("off")
    axes[1].imshow(base_img); axes[1].imshow(ig_map, alpha=0.5, cmap="jet")
    axes[1].set_title("ViT Integrated Gradients"); axes[1].axis("off")
    axes[2].imshow(base_img); axes[2].imshow(attn_map, alpha=0.5, cmap="jet")
    axes[2].set_title("ViT Attention Rollout"); axes[2].axis("off")
    plt.tight_layout()
    label = "correct" if k < 2 else "error"
    save_fig(f"vit_interpretability_{label}_{k+1}")

# ------------------------------------------------------------------
# Section 23 - Tabular interpretability
# ------------------------------------------------------------------
print("\n[Section 23] Tabular interpretability (Permutation, SHAP, PDP/ICE)")

# Permutation importance
plt.figure(figsize=(10, 6))
top_perm = perm_df.head(15).iloc[::-1]
plt.barh(top_perm["feature"], top_perm["importance_mean"])
plt.title("Permutation importance for the tabular model")
plt.xlabel("Mean drop in macro F1 after permutation")
plt.tight_layout()
save_fig("permutation_importance")

# SHAP bar chart
plt.figure(figsize=(10, 6))
top_shap = shap_importance.head(15).iloc[::-1]
plt.barh(top_shap["feature"], top_shap["mean_abs_shap"])
plt.title("SHAP feature importance for the tabular model")
plt.xlabel("Mean absolute SHAP value")
plt.tight_layout()
save_fig("shap_feature_importance")

# SHAP summary plot (beeswarm) - shap creates its own figure
if isinstance(shap_values, list):
    # For multiclass: use the class with highest average SHAP as the representative
    _sv = shap_values[0]
else:
    _sv_arr = np.array(shap_values)
    if _sv_arr.ndim == 3:
        _sv = _sv_arr[:, :, 0]
    else:
        _sv = _sv_arr

plt.figure(figsize=(10, 8))
shap.summary_plot(
    _sv,
    features=pd.DataFrame(X_shap, columns=feature_columns),
    feature_names=feature_columns,
    max_display=15,
    show=False,
)
plt.title("SHAP beeswarm plot")
plt.tight_layout()
save_fig("shap_beeswarm")

# PDP / ICE
top_feature_names = shap_importance["feature"].head(3).tolist()
target_class_for_pdp = 0
for i, name in enumerate(class_names):
    if "tumor" in name.lower() or "tumour" in name.lower():
        target_class_for_pdp = i
        break

X_tab_train_df = pd.DataFrame(X_tab_train, columns=feature_columns)
for j, feature_name in enumerate(top_feature_names):
    fig, ax = plt.subplots(figsize=(6, 4))
    PartialDependenceDisplay.from_estimator(
        rf_interpret_model, X_tab_train_df, [feature_name],
        target=target_class_for_pdp, kind="both", ax=ax,
    )
    ax.set_title(f"PDP + ICE for {feature_name}\n(target class: {class_names[target_class_for_pdp]})")
    plt.tight_layout()
    save_fig(f"pdp_ice_{feature_name}")

# ------------------------------------------------------------------
# Section 24 - Fusion interpretability
# ------------------------------------------------------------------
print("\n[Section 24] Fusion interpretability (Gates + Cross-attention)")

# Gate weights
plt.figure(figsize=(10, 5))
gate_summary = gate_df.groupby("true_class_name")[["image_gate", "tabular_gate"]].mean().sort_values("image_gate", ascending=False)
x_pos = np.arange(len(gate_summary))
plt.bar(x_pos - 0.2, gate_summary["image_gate"].values, width=0.4, label="image")
plt.bar(x_pos + 0.2, gate_summary["tabular_gate"].values, width=0.4, label="tabular")
plt.xticks(x_pos, gate_summary.index, rotation=45, ha="right")
plt.ylabel("Mean gate weight")
plt.title("Average gate weight by true class")
plt.legend()
plt.tight_layout()
save_fig("gated_fusion_weights")

# Cross-attention
plt.figure(figsize=(4, 4))
plt.imshow(mean_cross_attn, cmap="viridis")
plt.xticks([0, 1], ["image_token", "tabular_token"])
plt.yticks([0, 1], ["image_token", "tabular_token"])
plt.title("Mean cross-attention matrix")
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{mean_cross_attn[i, j]:.3f}", ha="center", va="center")
plt.colorbar()
plt.tight_layout()
save_fig("cross_attention_matrix")

# ------------------------------------------------------------------
# Section 25 - LIME (only if it was enabled)
# ------------------------------------------------------------------
if CONFIG.get("run_lime", False):
    print("\n[Section 25] LIME explanations")
    try:
        plt.figure(figsize=(6, 6))
        plt.imshow(mark_boundaries(temp_img / 255.0 if temp_img.max() > 1 else temp_img, temp_mask))
        plt.title("LIME image explanation (ResNet18)")
        plt.axis("off")
        plt.tight_layout()
        save_fig("lime_image_explanation")
    except NameError:
        print("  LIME variables not found, skipping.")
else:
    print("\n[Section 25] LIME was disabled (CONFIG['run_lime'] = False), skipping.")

# ------------------------------------------------------------------
# Section 26 - Error analysis
# ------------------------------------------------------------------
print("\n[Section 26] Error analysis")

# Per-class accuracy
plt.figure(figsize=(10, 4))
plt.bar(per_class_df["class_name"], per_class_df["class_accuracy"])
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.title(f"Per-class accuracy for the best model: {best_model_name}")
plt.tight_layout()
save_fig("per_class_accuracy")

# Confident errors
confident_errors = error_df.query("correct == 0").sort_values("confidence", ascending=False).head(4)
for k, (_, row) in enumerate(confident_errors.iterrows()):
    idx = int(row["global_index"])
    img = images[idx]
    hematoxylin, smooth, mask, labels_ws = segment_nuclei_from_histology(img)
    boxes = labels_to_boxes(labels_ws)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(img)
    axes[0].set_title(f"Confident error\nTrue: {row['true_class']}\nPred: {row['pred_class']} ({row['confidence']:.3f})")
    axes[0].axis("off")
    axes[1].imshow(mask, cmap="gray"); axes[1].set_title("Segmentation mask"); axes[1].axis("off")
    axes[2].imshow(img)
    for box in boxes[:30]:
        rect = plt.Rectangle((box["x1"], box["y1"]), box["x2"]-box["x1"], box["y2"]-box["y1"],
                              fill=False, linewidth=1.0)
        axes[2].add_patch(rect)
    axes[2].set_title(f"Detection ({len(boxes)} boxes)"); axes[2].axis("off")
    plt.tight_layout()
    save_fig(f"error_analysis_confident_error_{k+1}")

# Uncertain correct
uncertain_correct = error_df.query("correct == 1").sort_values("confidence", ascending=True).head(4)
for k, (_, row) in enumerate(uncertain_correct.iterrows()):
    idx = int(row["global_index"])
    img = images[idx]
    hematoxylin, smooth, mask, labels_ws = segment_nuclei_from_histology(img)
    boxes = labels_to_boxes(labels_ws)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].imshow(img)
    axes[0].set_title(f"Uncertain correct\nTrue: {row['true_class']}\nPred: {row['pred_class']} ({row['confidence']:.3f})")
    axes[0].axis("off")
    axes[1].imshow(mask, cmap="gray"); axes[1].set_title("Segmentation mask"); axes[1].axis("off")
    axes[2].imshow(img)
    for box in boxes[:30]:
        rect = plt.Rectangle((box["x1"], box["y1"]), box["x2"]-box["x1"], box["y2"]-box["y1"],
                              fill=False, linewidth=1.0)
        axes[2].add_patch(rect)
    axes[2].set_title(f"Detection ({len(boxes)} boxes)"); axes[2].axis("off")
    plt.tight_layout()
    save_fig(f"error_analysis_uncertain_correct_{k+1}")

# ------------------------------------------------------------------
# Section 27 - Detailed multimodal case study
# ------------------------------------------------------------------
print("\n[Section 27] Multimodal case study")

case_idx = int(confident_errors.iloc[0]["global_index"]) if len(confident_errors) > 0 else int(test_idx[0])

# Case study overview (original + seg + detection)
show_test_case(case_idx, title_prefix="Case study | ")
save_fig("case_study_overview")

# Local SHAP
local_shap_df = get_local_shap_for_sample(case_idx)
plt.figure(figsize=(8, 5))
top_local = local_shap_df.head(10).iloc[::-1]
plt.barh(top_local["feature"], top_local["shap_value"])
plt.title("Local tabular SHAP explanation")
plt.xlabel("Contribution toward the chosen class score")
plt.tight_layout()
save_fig("case_study_local_shap")

# Case study ResNet explanations
info = explain_resnet_sample(case_idx)
save_fig("case_study_resnet_explanations")

# Case study ViT explanations
explain_vit_sample(case_idx)
save_fig("case_study_vit_explanations")

# ------------------------------------------------------------------
# ZIP and download
# ------------------------------------------------------------------
print("\n" + "=" * 60)
zip_name = "assignment1_figures.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(FIGURE_DIR)):
        zf.write(os.path.join(FIGURE_DIR, fname), fname)

total = fig_counter[0]
print(f"Done! Saved {total} figures to '{FIGURE_DIR}/'")
print(f"Zipped into '{zip_name}'")
print("=" * 60)

colab_files.download(zip_name)


## Save results CSV

In [ ]:
results_df.to_csv("tcga_brca_results.csv", index=False)
display(results_df)
print("Pipeline complete.")
